# CMIP6 Tmax < Tmin investigation, first version (superseded)

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Detailed investigation of the Tmax < Tmin reversals and the −273.15 °C sentinel values in the SSP2-4.5 files: grid-cell audits, spatial and temporal patterns, clustering, cross-model synchronisation and checks against the original source files. The two findings still hold for the dataset (0 K fill values; small reversals from the univariate bias correction, mainly CanESM5, INM-CM4-8 and INM-CM5-0), but the cells examined were the offset ones.

> ⚠️ **Superseded — kept as a record of the investigation.** These cells read grid columns at `3 + column_index_zero_based`, which are about 0.75° north of Alwar (the index already counts the three date columns). The corrected data are produced by `02_climate/03_climate_forcing_corrected.ipynb`. Do not use outputs of this notebook as model input.

| | |
|---|---|
| **Reads** | `Vimal_Mishra_CMIP6/SSP245_extracted/India/…`, `Alwar_grid_analysis/Alwar_daily_extracted_REBUILT/` |
| **Writes** | `Alwar_grid_analysis/Alwar_block_daily_forcing_REBUILT/`, `Alwar_grid_analysis/Climate_Indicators/…` |
| **Cells** | 22 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Rebuild block forcing from the `REBUILT` cell files
- 2. Grid-cell and source-level audits
- 3. Where and when the reversals occur
- 4. Cross-model synchronisation
- 5. Checks against the original source; sentinel values

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["geopandas"], modflow=False)
del TOKEN, URL


## 1. Rebuild block forcing from the `REBUILT` cell files

**[orig #218]** Reconstruct block-daily forcing for the 14 blocks from the 39 `REBUILT` cell files  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ================================================================
# RECONSTRUCT DAILY CLIMATE FORCING FOR 14 ALWAR BLOCKS
#
# Inputs:
# 39 rebuilt CMIP6 files
# 22 spatial grid cells
# Existing area-weighted spatial intersections
#
# Outputs:
# 1. Daily weighted block datasets (original dates)
# 2. Calendar-completed block datasets
# 3. Leap-day interpolation audit
# 4. Spatial weighting validation report
#
# No original data are overwritten.
# ================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(
    BASE_DIR, "Alwar_grid_analysis"
)

INPUT_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_daily_extracted_REBUILT"
)

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

RAW_OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Alwar_block_daily_weighted_REBUILT"
)

COMPLETED_OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Alwar_block_daily_forcing_REBUILT"
)

REPORT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Climate_Indicators"
)

os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPLETED_OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. MODELS AND VARIABLES
# ------------------------------------------------

models = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

variables = [
    "PrecipData",
    "TMaxData",
    "TMinData"
]

# ------------------------------------------------
# 3. READ SPATIAL WEIGHTS
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

weights["block_lgd"] = (
    weights["block_lgd"].astype(int)
)

weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)

weights["block_name"] = (
    weights["block_name"].astype(str).str.strip()
)

weight_column = "weight_within_intersected_area"

required_weight_columns = [
    "block_lgd",
    "block_name",
    "column_index_zero_based",
    weight_column
]

for col in required_weight_columns:
    if col not in weights.columns:
        raise ValueError(f"Missing weight field: {col}")

# Check weights sum to 1 for each block
weight_sum = weights.groupby("block_lgd")[
    weight_column
].sum()

if not np.allclose(
    weight_sum.values,
    1.0,
    atol=1e-6
):
    raise ValueError(
        "Spatial weights do not sum to 1 for every block."
    )

blocks = (
    weights[["block_lgd", "block_name"]]
    .drop_duplicates()
    .sort_values("block_lgd")
)

if len(blocks) != 14:
    raise ValueError(
        f"Expected 14 blocks, found {len(blocks)}"
    )

print("Blocks identified:", len(blocks))
print("Spatial weight validation: PASS")

# ------------------------------------------------
# 4. EXPECTED DAILY CALENDAR
# ------------------------------------------------

expected_dates = pd.date_range(
    start="2015-01-01",
    end="2100-12-31",
    freq="D"
)

print("Expected calendar days:", len(expected_dates))

# ------------------------------------------------
# 5. PROCESS ALL 39 FILES
# ------------------------------------------------

leap_audit = []
spatial_audit = []
file_inventory = []

for model in models:

    for variable in variables:

        print(
            f"\nProcessing {model} | {variable}"
        )

        input_filename = (
            f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
        )

        input_path = os.path.join(
            INPUT_DIR,
            input_filename
        )

        if not os.path.exists(input_path):
            raise FileNotFoundError(input_path)

        df = pd.read_csv(
            input_path,
            parse_dates=["Date"]
        )

        # ------------------------------------------------
        # 5A. CHECK GRID COLUMNS
        # ------------------------------------------------

        model_weights = weights.copy()

        grid_columns = sorted(
            {
                f"grid_col_{i}"
                for i in model_weights[
                    "column_index_zero_based"
                ].unique()
            }
        )

        missing_columns = [
            col for col in grid_columns
            if col not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing grid cells in {input_filename}: "
                f"{missing_columns}"
            )

        # ------------------------------------------------
        # 5B. CHECK DATE SEQUENCE
        # ------------------------------------------------

        if df["Date"].duplicated().any():
            raise ValueError(
                f"Duplicate dates in {input_filename}"
            )

        if not df["Date"].is_monotonic_increasing:
            raise ValueError(
                f"Dates not chronological in {input_filename}"
            )

        if df["Date"].min() != expected_dates.min():
            raise ValueError(
                f"Unexpected first date in {input_filename}"
            )

        if df["Date"].max() != expected_dates.max():
            raise ValueError(
                f"Unexpected last date in {input_filename}"
            )

        missing_dates = expected_dates.difference(
            pd.DatetimeIndex(df["Date"])
        )

        unexpected_dates = pd.DatetimeIndex(
            df["Date"]
        ).difference(expected_dates)

        # Only missing February 29 dates are permitted
        non_leap_missing = [
            d for d in missing_dates
            if not (d.month == 2 and d.day == 29)
        ]

        if len(non_leap_missing) > 0:
            raise ValueError(
                f"Unexpected missing dates in {input_filename}: "
                f"{non_leap_missing[:10]}"
            )

        if len(unexpected_dates) > 0:
            raise ValueError(
                f"Unexpected dates in {input_filename}"
            )

        # ------------------------------------------------
        # 5C. CALCULATE DAILY AREA-WEIGHTED BLOCK VALUES
        # ------------------------------------------------

        block_output = pd.DataFrame({
            "Date": df["Date"]
        })

        for _, block in blocks.iterrows():

            lgd = int(block["block_lgd"])
            block_name = block["block_name"]

            block_column = f"{lgd}_{block_name}"

            block_weights = weights[
                weights["block_lgd"] == lgd
            ].copy()

            cell_columns = [
                f"grid_col_{i}"
                for i in block_weights[
                    "column_index_zero_based"
                ]
            ]

            values = df[cell_columns].to_numpy(
                dtype=float
            )

            w = block_weights[
                weight_column
            ].to_numpy(dtype=float)

            # Do not silently ignore missing cells.
            if not np.isfinite(values).all():
                raise ValueError(
                    f"Non-finite grid data for "
                    f"{model}, {variable}, block {block_name}"
                )

            if not np.isclose(w.sum(), 1.0, atol=1e-6):
                raise ValueError(
                    f"Weights do not sum to 1 for {block_name}"
                )

            weighted_values = values @ w

            block_output[block_column] = weighted_values

        # ------------------------------------------------
        # 5D. SAVE ORIGINAL WEIGHTED DATA
        # ------------------------------------------------

        block_output["Year"] = (
            block_output["Date"].dt.year
        )

        block_output["Month"] = (
            block_output["Date"].dt.month
        )

        block_output["Day"] = (
            block_output["Date"].dt.day
        )

        block_columns = [
            f"{int(row.block_lgd)}_{row.block_name}"
            for row in blocks.itertuples()
        ]

        block_output = block_output[
            ["Date", "Year", "Month", "Day"]
            + block_columns
        ]

        raw_filename = (
            f"{model}_ssp245_{variable}_AlwarBlocks_WEIGHTED.csv"
        )

        raw_path = os.path.join(
            RAW_OUTPUT_DIR,
            raw_filename
        )

        block_output.to_csv(
            raw_path,
            index=False
        )

        # ------------------------------------------------
        # 5E. COMPLETE MISSING FEBRUARY 29 DATES
        # ------------------------------------------------

        completed = (
            block_output
            .set_index("Date")
            .reindex(expected_dates)
        )

        inserted_dates = []

        for missing_date in missing_dates:

            previous_date = (
                missing_date - pd.Timedelta(days=1)
            )

            next_date = (
                missing_date + pd.Timedelta(days=1)
            )

            for col in block_columns:

                previous_value = completed.loc[
                    previous_date, col
                ]

                next_value = completed.loc[
                    next_date, col
                ]

                if not np.isfinite(previous_value):
                    raise ValueError(
                        f"Missing Feb 28 value: {model}, "
                        f"{variable}, {col}"
                    )

                if not np.isfinite(next_value):
                    raise ValueError(
                        f"Missing Mar 1 value: {model}, "
                        f"{variable}, {col}"
                    )

                completed.loc[
                    missing_date, col
                ] = (
                    previous_value + next_value
                ) / 2.0

            inserted_dates.append(missing_date)

            leap_audit.append({
                "Model": model,
                "Variable": variable,
                "Interpolated_Date": missing_date,
                "Method": (
                    "Arithmetic midpoint of February 28 "
                    "and March 1"
                ),
                "Number_of_Blocks_Interpolated": len(block_columns),
                "Status": "INTERPOLATED"
            })

        # ------------------------------------------------
        # 5F. FINAL CALENDAR VALIDATION
        # ------------------------------------------------

        completed = completed.reset_index()

        completed = completed.rename(
            columns={"index": "Date"}
        )

        completed["Year"] = (
            completed["Date"].dt.year
        )

        completed["Month"] = (
            completed["Date"].dt.month
        )

        completed["Day"] = (
            completed["Date"].dt.day
        )

        completed = completed[
            ["Date", "Year", "Month", "Day"]
            + block_columns
        ]

        if len(completed) != len(expected_dates):
            raise ValueError(
                f"Incorrect completed row count: {model} {variable}"
            )

        if completed.isna().any().any():
            raise ValueError(
                f"Missing values remain after calendar completion: "
                f"{model} {variable}"
            )

        if not completed["Date"].equals(
            pd.Series(expected_dates, name="Date")
        ):
            raise ValueError(
                f"Final dates do not match expected calendar: "
                f"{model} {variable}"
            )

        # ------------------------------------------------
        # 5G. SAVE COMPLETED BLOCK DATA
        # ------------------------------------------------

        completed_filename = (
            f"{model}_ssp245_{variable}_AlwarBlocks.csv"
        )

        completed_path = os.path.join(
            COMPLETED_OUTPUT_DIR,
            completed_filename
        )

        completed.to_csv(
            completed_path,
            index=False
        )

        # ------------------------------------------------
        # 5H. RECORD AUDITS
        # ------------------------------------------------

        leap_audit.append({
            "Model": model,
            "Variable": variable,
            "Interpolated_Date": "SUMMARY",
            "Method": "Calendar audit",
            "Number_of_Blocks_Interpolated": len(block_columns),
            "Status": (
                f"{len(inserted_dates)} leap days inserted"
            )
        })

        spatial_audit.append({
            "Model": model,
            "Variable": variable,
            "Original_Days": len(block_output),
            "Missing_Leap_Days": len(missing_dates),
            "Rebuilt_Days": len(completed),
            "Blocks": len(block_columns),
            "Status": "PASS"
        })

        file_inventory.append({
            "Model": model,
            "Variable": variable,
            "Raw_File": raw_filename,
            "Completed_File": completed_filename,
            "Raw_Rows": len(block_output),
            "Completed_Rows": len(completed),
            "Status": "PASS"
        })

# ------------------------------------------------
# 6. SAVE AUDIT REPORTS
# ------------------------------------------------

leap_audit_df = pd.DataFrame(leap_audit)

spatial_audit_df = pd.DataFrame(spatial_audit)

inventory_df = pd.DataFrame(file_inventory)

leap_audit_path = os.path.join(
    REPORT_DIR,
    "Rebuilt_Leap_Day_Interpolation_Audit.csv"
)

spatial_audit_path = os.path.join(
    REPORT_DIR,
    "Rebuilt_Block_Spatial_Extraction_Audit.csv"
)

inventory_path = os.path.join(
    REPORT_DIR,
    "Rebuilt_Block_Forcing_Inventory.csv"
)

leap_audit_df.to_csv(
    leap_audit_path,
    index=False
)

spatial_audit_df.to_csv(
    spatial_audit_path,
    index=False
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

# ------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("BLOCK-LEVEL CLIMATE RECONSTRUCTION SUMMARY")
print("=" * 70)

print("Models:", len(models))
print("Variables:", len(variables))
print("Expected files:", len(models) * len(variables))

print("\nSpatial extraction audit:")
display(spatial_audit_df)

print("\nLeap-day audit summary:")
display(
    leap_audit_df[
        leap_audit_df["Interpolated_Date"] == "SUMMARY"
    ]
)

print("\nInventory:")
display(inventory_df)

print("\nFiles successfully completed:", len(inventory_df))

print("\nRaw weighted data directory:")
print(RAW_OUTPUT_DIR)

print("\nCalendar-completed data directory:")
print(COMPLETED_OUTPUT_DIR)

print("\nAudit reports:")
print(leap_audit_path)
print(spatial_audit_path)
print(inventory_path)

if len(inventory_df) != 39:
    raise RuntimeError("Expected 39 completed block files.")

if not (inventory_df["Status"] == "PASS").all():
    raise RuntimeError("Some block files failed validation.")

print("\nSUCCESS: All 39 block-level climate datasets reconstructed.")

## 2. Grid-cell and source-level audits

**[orig #219]** Grid-cell Tmax/Tmin consistency audit (13 models × 22 cells)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ================================================================
# GRID-CELL LEVEL TMAX / TMIN CONSISTENCY AUDIT
# 13 CMIP6 MODELS | 22 ALWAR GRID CELLS
# ================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

GRID_DIR = os.path.join(
    BASE_DIR,
    "Alwar_grid_analysis",
    "Alwar_daily_extracted_REBUILT"
)

WEIGHT_FILE = os.path.join(
    BASE_DIR,
    "Alwar_grid_analysis",
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

REPORT_DIR = os.path.join(
    BASE_DIR,
    "Alwar_grid_analysis",
    "Climate_Indicators"
)

os.makedirs(REPORT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. MODELS
# ------------------------------------------------

models = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

# ------------------------------------------------
# 3. READ REQUIRED GRID INDICES
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

grid_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

grid_columns = [
    f"grid_col_{i}"
    for i in grid_indices
]

print("Number of grid cells:", len(grid_columns))

# ------------------------------------------------
# 4. AUDIT ALL MODELS
# ------------------------------------------------

detail_records = []
summary_records = []
extreme_records = []

for model in models:

    print("\nAuditing:", model)

    tmax_path = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMaxData_Alwar_REBUILT.csv"
    )

    tmin_path = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMinData_Alwar_REBUILT.csv"
    )

    if not os.path.isfile(tmax_path):
        raise FileNotFoundError(tmax_path)

    if not os.path.isfile(tmin_path):
        raise FileNotFoundError(tmin_path)

    tmax = pd.read_csv(
        tmax_path,
        parse_dates=["Date"]
    )

    tmin = pd.read_csv(
        tmin_path,
        parse_dates=["Date"]
    )

    # ------------------------------------------------
    # 4A. DATE ALIGNMENT
    # ------------------------------------------------

    if not tmax["Date"].equals(tmin["Date"]):
        raise ValueError(
            f"Date mismatch between Tmax and Tmin: {model}"
        )

    for col in grid_columns:
        if col not in tmax.columns:
            raise KeyError(f"{col} missing in Tmax: {model}")

        if col not in tmin.columns:
            raise KeyError(f"{col} missing in Tmin: {model}")

    # ------------------------------------------------
    # 4B. GRID-CELL TEMPERATURE COMPARISON
    # ------------------------------------------------

    total_violations = 0
    total_pairs = 0
    total_extreme = 0

    for col, grid_index in zip(
        grid_columns,
        grid_indices
    ):

        xmax = tmax[col].to_numpy(dtype=float)
        xmin = tmin[col].to_numpy(dtype=float)

        valid = (
            np.isfinite(xmax)
            & np.isfinite(xmin)
        )

        violation = valid & (xmax < xmin)

        extreme = valid & (
            (xmax < -30)
            | (xmin < -30)
            | (xmax > 60)
            | (xmin > 60)
        )

        total_pairs += int(valid.sum())
        total_violations += int(violation.sum())
        total_extreme += int(extreme.sum())

        # Record each grid cell
        detail_records.append({
            "Model": model,
            "Grid_Index": grid_index,
            "Grid_Column": col,
            "Valid_Pairs": int(valid.sum()),
            "Tmax_Below_Tmin_Count": int(violation.sum()),
            "Violation_Percent": (
                100 * violation.sum() / valid.sum()
                if valid.sum() else np.nan
            ),
            "Extreme_Temperature_Count": int(extreme.sum()),
            "Tmax_Min": float(np.nanmin(xmax)),
            "Tmax_Max": float(np.nanmax(xmax)),
            "Tmin_Min": float(np.nanmin(xmin)),
            "Tmin_Max": float(np.nanmax(xmin))
        })

        # Record every violation
        positions = np.where(violation)[0]

        for pos in positions:

            detail = {
                "Model": model,
                "Date": tmax.iloc[pos]["Date"],
                "Grid_Index": grid_index,
                "Grid_Column": col,
                "Tmax_C": xmax[pos],
                "Tmin_C": xmin[pos],
                "Tmin_minus_Tmax_C": (
                    xmin[pos] - xmax[pos]
                )
            }

            extreme_records.append(detail)

    # ------------------------------------------------
    # 4C. MODEL SUMMARY
    # ------------------------------------------------

    summary_records.append({
        "Model": model,
        "Total_Valid_Grid_Day_Pairs": total_pairs,
        "Tmax_Below_Tmin_Count": total_violations,
        "Violation_Percent": (
            100 * total_violations / total_pairs
            if total_pairs else np.nan
        ),
        "Extreme_Temperature_Count": total_extreme,
        "Status": (
            "VIOLATIONS_FOUND"
            if total_violations > 0
            else "PASS"
        )
    })

# ------------------------------------------------
# 5. CREATE REPORTS
# ------------------------------------------------

grid_detail = pd.DataFrame(detail_records)

model_summary = pd.DataFrame(summary_records)

violation_detail = pd.DataFrame(extreme_records)

# ------------------------------------------------
# 6. SORT VIOLATIONS
# ------------------------------------------------

if not violation_detail.empty:

    violation_detail = violation_detail.sort_values(
        "Tmin_minus_Tmax_C",
        ascending=False
    )

# ------------------------------------------------
# 7. SAVE REPORTS
# ------------------------------------------------

grid_detail_path = os.path.join(
    REPORT_DIR,
    "Grid_Cell_Temperature_Consistency_Detail.csv"
)

summary_path = os.path.join(
    REPORT_DIR,
    "Grid_Cell_Temperature_Consistency_Model_Summary.csv"
)

violation_path = os.path.join(
    REPORT_DIR,
    "Grid_Cell_Tmax_Below_Tmin_Violations.csv"
)

grid_detail.to_csv(
    grid_detail_path,
    index=False
)

model_summary.to_csv(
    summary_path,
    index=False
)

violation_detail.to_csv(
    violation_path,
    index=False
)

# ------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("GRID-CELL TEMPERATURE CONSISTENCY RESULTS")
print("=" * 70)

print("\nMODEL SUMMARY")
display(model_summary)

print("\nGRID-CELL DETAILS")
display(
    grid_detail.sort_values(
        "Tmax_Below_Tmin_Count",
        ascending=False
    ).head(30)
)

print("\nNUMBER OF VIOLATIONS:", len(violation_detail))

print("\nLARGEST VIOLATIONS")
if not violation_detail.empty:
    display(violation_detail.head(30))
else:
    print("No Tmax < Tmin cases found.")

# ------------------------------------------------
# 9. SAVE SUMMARY STATISTICS
# ------------------------------------------------

print("\nREPORTS SAVED:")
print(grid_detail_path)
print(summary_path)
print(violation_path)

print("\nGRID-CELL AUDIT COMPLETE.")

**[orig #220]** Source-level investigation of Tmax < Tmin violations (SSP2-4.5)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# SOURCE-LEVEL INVESTIGATION OF CMIP6 Tmax/Tmin VIOLATIONS
# Study area: Alwar district, Rajasthan
# Scenario: SSP245
#
# Purpose:
# 1. Read all grid-cell Tmax < Tmin violations.
# 2. Trace each flagged observation to the original source files.
# 3. Compare original values with extracted values.
# 4. Identify zero values and possible Kelvin-to-Celsius issues.
# 5. Save a detailed source-tracing report.
#
# IMPORTANT:
# No observations are modified or corrected.
# ================================================================

import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

# -------------------- PATHS --------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

GRID_DIR = os.path.join(
    BASE_DIR, "Alwar_daily_extracted_REBUILT"
)

SOURCE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

INDICATOR_DIR = os.path.join(
    BASE_DIR, "Climate_Indicators"
)

VIOLATION_FILE = os.path.join(
    INDICATOR_DIR,
    "Grid_Cell_Tmax_Below_Tmin_Violations.csv"
)

OUTPUT_FILE = os.path.join(
    INDICATOR_DIR,
    "Tmax_Tmin_Original_Source_Trace.csv"
)

SUMMARY_FILE = os.path.join(
    INDICATOR_DIR,
    "Tmax_Tmin_Source_Trace_Summary.csv"
)

os.makedirs(INDICATOR_DIR, exist_ok=True)

# -------------------- MODELS --------------------

MODELS = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

# -------------------- CHECK INPUT --------------------

if not os.path.exists(VIOLATION_FILE):
    raise FileNotFoundError(
        f"Violation report not found: {VIOLATION_FILE}"
    )

violations = pd.read_csv(VIOLATION_FILE)

print("Violation report columns:")
print(violations.columns.tolist())

# Standardize date and grid index
violations["Date"] = pd.to_datetime(
    violations["Date"]
)

violations["Grid_Index"] = pd.to_numeric(
    violations["Grid_Index"],
    errors="coerce"
).astype("Int64")

print("\nTotal violations to investigate:", len(violations))

# -------------------- SOURCE FILE LOCATOR --------------------

def find_source_file(model, variable):

    pattern = os.path.join(
        SOURCE_DIR,
        "**",
        f"*{model}*ssp245*{variable}*"
    )

    matches = glob.glob(
        pattern,
        recursive=True
    )

    # Exclude directories
    matches = [
        f for f in matches
        if os.path.isfile(f)
    ]

    # Exact model-token filtering avoids matching
    # EC-Earth3 with EC-Earth3-Veg.
    exact_matches = []

    for f in matches:
        name = os.path.basename(f)

        if model == "EC-Earth3":
            if "EC-Earth3-Veg" in name:
                continue

        if model == "EC-Earth3-Veg":
            if "EC-Earth3-Veg" not in name:
                continue

        exact_matches.append(f)

    if len(exact_matches) == 0:
        raise FileNotFoundError(
            f"No source file found for {model}, {variable}"
        )

    if len(exact_matches) > 1:
        raise RuntimeError(
            f"Multiple source files found for {model}, "
            f"{variable}: {exact_matches}"
        )

    return exact_matches[0]


# -------------------- READ ORIGINAL SOURCE --------------------

def read_original_source(filepath):

    # The first two rows contain longitude and latitude.
    # Daily data starts after those two metadata rows.
    raw = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        skiprows=2,
        engine="python"
    )

    if raw.shape[1] < 4:
        raise ValueError(
            f"Unexpected source format: {filepath}"
        )

    raw = raw.apply(
        pd.to_numeric,
        errors="coerce"
    )

    raw.columns = (
        ["Year", "Month", "Day"]
        + [
            f"grid_col_{i}"
            for i in range(raw.shape[1] - 3)
        ]
    )

    raw["Date"] = pd.to_datetime(
        dict(
            year=raw["Year"],
            month=raw["Month"],
            day=raw["Day"]
        ),
        errors="coerce"
    )

    return raw


# -------------------- CACHE SOURCE DATA --------------------

source_cache = {}

def get_source(model, variable):

    key = (model, variable)

    if key not in source_cache:

        filepath = find_source_file(
            model,
            variable
        )

        print(
            f"Reading original source: "
            f"{model} | {variable}"
        )

        source_cache[key] = (
            read_original_source(filepath)
        )

    return source_cache[key]


# -------------------- READ EXTRACTED DATA --------------------

def read_extracted(model, variable):

    filepath = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
    )

    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)

    df = pd.read_csv(filepath)

    df["Date"] = pd.to_datetime(
        df["Date"]
    )

    return df


# -------------------- TRACE EACH VIOLATION --------------------

results = []

models_to_process = sorted(
    violations["Model"].dropna().unique()
)

for model in models_to_process:

    print("\nInvestigating:", model)

    model_violations = violations[
        violations["Model"] == model
    ].copy()

    if model_violations.empty:
        continue

    tmax_source = get_source(
        model, "TMaxData"
    )

    tmin_source = get_source(
        model, "TMinData"
    )

    tmax_extracted = read_extracted(
        model, "TMaxData"
    )

    tmin_extracted = read_extracted(
        model, "TMinData"
    )

    # Index source and extracted data by date
    tmax_source = tmax_source.set_index("Date")
    tmin_source = tmin_source.set_index("Date")

    tmax_extracted = tmax_extracted.set_index("Date")
    tmin_extracted = tmin_extracted.set_index("Date")

    for _, event in model_violations.iterrows():

        date = event["Date"]
        grid_index = int(event["Grid_Index"])

        col = f"grid_col_{grid_index}"

        record = {
            "Model": model,
            "Date": date,
            "Grid_Index": grid_index,
            "Grid_Column": col,
            "Reported_Tmax_C": event["Tmax_C"],
            "Reported_Tmin_C": event["Tmin_C"],
            "Reported_Reversal_C": event[
                "Tmin_minus_Tmax_C"
            ]
        }

        # Extracted values
        if (
            date in tmax_extracted.index
            and col in tmax_extracted.columns
            and date in tmin_extracted.index
            and col in tmin_extracted.columns
        ):

            extracted_tmax = float(
                tmax_extracted.loc[date, col]
            )

            extracted_tmin = float(
                tmin_extracted.loc[date, col]
            )

        else:
            extracted_tmax = np.nan
            extracted_tmin = np.nan

        # Original source values
        if (
            date in tmax_source.index
            and col in tmax_source.columns
            and date in tmin_source.index
            and col in tmin_source.columns
        ):

            original_tmax = float(
                tmax_source.loc[date, col]
            )

            original_tmin = float(
                tmin_source.loc[date, col]
            )

        else:
            original_tmax = np.nan
            original_tmin = np.nan

        record.update({
            "Extracted_Tmax": extracted_tmax,
            "Extracted_Tmin": extracted_tmin,
            "Original_Tmax_Raw": original_tmax,
            "Original_Tmin_Raw": original_tmin,
            "Original_Raw_Tmax_Minus_Tmin":
                original_tmax - original_tmin
                if np.isfinite(original_tmax)
                and np.isfinite(original_tmin)
                else np.nan
        })

        # Exact source/extraction comparison
        record["Tmax_Extraction_Difference"] = (
            extracted_tmax - original_tmax
            if np.isfinite(extracted_tmax)
            and np.isfinite(original_tmax)
            else np.nan
        )

        record["Tmin_Extraction_Difference"] = (
            extracted_tmin - original_tmin
            if np.isfinite(extracted_tmin)
            and np.isfinite(original_tmin)
            else np.nan
        )

        # Diagnostic flags
        record["Original_Tmax_Is_Zero"] = (
            np.isclose(original_tmax, 0)
            if np.isfinite(original_tmax)
            else False
        )

        record["Original_Tmin_Is_Zero"] = (
            np.isclose(original_tmin, 0)
            if np.isfinite(original_tmin)
            else False
        )

        record["Original_Source_Reversal"] = (
            original_tmax < original_tmin
            if np.isfinite(original_tmax)
            and np.isfinite(original_tmin)
            else False
        )

        record["Possible_Zero_Kelvin_Fill"] = (
            np.isclose(extracted_tmax, -273.15)
            or np.isclose(extracted_tmin, -273.15)
        )

        results.append(record)


# -------------------- SAVE RESULTS --------------------

trace = pd.DataFrame(results)

trace.to_csv(
    OUTPUT_FILE,
    index=False
)

# -------------------- SUMMARY --------------------

summary = (
    trace.groupby("Model")
    .agg(
        Events=("Date", "count"),
        Original_Source_Reversals=(
            "Original_Source_Reversal", "sum"
        ),
        Tmax_Extraction_Differences=(
            "Tmax_Extraction_Difference",
            lambda x: (x.abs() > 1e-8).sum()
        ),
        Tmin_Extraction_Differences=(
            "Tmin_Extraction_Difference",
            lambda x: (x.abs() > 1e-8).sum()
        ),
        Original_Tmax_Zero_Count=(
            "Original_Tmax_Is_Zero", "sum"
        ),
        Original_Tmin_Zero_Count=(
            "Original_Tmin_Is_Zero", "sum"
        ),
        Possible_Zero_Kelvin_Count=(
            "Possible_Zero_Kelvin_Fill", "sum"
        )
    )
    .reset_index()
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

# -------------------- DISPLAY --------------------

print("\n" + "=" * 70)
print("ORIGINAL SOURCE TRACE SUMMARY")
print("=" * 70)

display(summary)

print("\nLargest source-level reversals:")

display(
    trace.sort_values(
        "Reported_Reversal_C",
        ascending=False
    ).head(20)
)

print("\nFiles saved:")
print(OUTPUT_FILE)
print(SUMMARY_FILE)

print("\nSOURCE-LEVEL INVESTIGATION COMPLETE.")


**[orig #224]** Original-source temperature violation investigation  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# =====================================================================
# CMIP6 ORIGINAL-SOURCE TEMPERATURE VIOLATION INVESTIGATION
# Study area: Alwar district, Rajasthan
# Scenario: SSP245
#
# Investigates all grid-cell Tmax < Tmin violations.
#
# Original source structure:
# India / MODEL / ssp245 / TMaxData
# India / MODEL / ssp245 / TMinData
#
# Reads only the required grid columns to reduce memory use.
# Does not modify any climate data.
# =====================================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------
# 1. PATHS
# ---------------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

SOURCE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

GRID_DIR = os.path.join(
    BASE_DIR,
    "Alwar_daily_extracted_REBUILT"
)

INDICATOR_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators"
)

VIOLATION_FILE = os.path.join(
    INDICATOR_DIR,
    "Grid_Cell_Tmax_Below_Tmin_Violations.csv"
)

OUTPUT_FILE = os.path.join(
    INDICATOR_DIR,
    "Tmax_Tmin_Original_Source_Trace.csv"
)

SUMMARY_FILE = os.path.join(
    INDICATOR_DIR,
    "Tmax_Tmin_Source_Trace_Summary.csv"
)

os.makedirs(INDICATOR_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# 2. CHECK INPUT FILES
# ---------------------------------------------------------------------

if not os.path.isfile(VIOLATION_FILE):
    raise FileNotFoundError(VIOLATION_FILE)

if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError(SOURCE_DIR)

violations = pd.read_csv(VIOLATION_FILE)

required_columns = [
    "Model",
    "Date",
    "Grid_Index",
    "Grid_Column",
    "Tmax_C",
    "Tmin_C",
    "Tmin_minus_Tmax_C"
]

missing = [
    c for c in required_columns
    if c not in violations.columns
]

if missing:
    raise ValueError(
        f"Missing columns in violation report: {missing}"
    )

violations["Date"] = pd.to_datetime(
    violations["Date"],
    errors="raise"
)

violations["Grid_Index"] = pd.to_numeric(
    violations["Grid_Index"],
    errors="raise"
).astype(int)

print("Total violations:", len(violations))
print("Models requiring investigation:")
print(sorted(violations["Model"].unique()))

# ---------------------------------------------------------------------
# 3. EXACT ORIGINAL SOURCE FILE PATH
# ---------------------------------------------------------------------

def source_filepath(model, variable):

    filepath = os.path.join(
        SOURCE_DIR,
        model,
        "ssp245",
        variable
    )

    if not os.path.isfile(filepath):
        raise FileNotFoundError(
            f"Original source file not found:\n{filepath}"
        )

    return filepath


# ---------------------------------------------------------------------
# 4. READ ONLY REQUIRED GRID COLUMNS FROM ORIGINAL SOURCE
# ---------------------------------------------------------------------

def read_selected_source(
    filepath,
    grid_indices,
    date_values
):
    """
    Source layout:
    First two rows: longitude and latitude metadata.
    Remaining rows:
    Year Month Day grid_cell_0 grid_cell_1 ...

    Grid index 1672 corresponds to source data column 3 + 1672.
    """

    grid_indices = sorted(
        set(int(x) for x in grid_indices)
    )

    # Select date columns and required grid columns only.
    usecols = (
        [0, 1, 2]
        + [3 + i for i in grid_indices]
    )

    names = (
        ["Year", "Month", "Day"]
        + [f"grid_col_{i}" for i in grid_indices]
    )

    print(
        "\nReading:",
        os.path.basename(filepath),
        "| Required grid cells:",
        len(grid_indices)
    )

    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        skiprows=2,
        usecols=usecols,
        names=names,
        engine="c"
    )

    # Convert selected columns to numeric
    for col in names:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df["Date"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        ),
        errors="coerce"
    )

    if df["Date"].isna().any():
        raise ValueError(
            f"Invalid dates found in original source: {filepath}"
        )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates found in original source: {filepath}"
        )

    # Retain only dates needed for this investigation.
    requested_dates = pd.DatetimeIndex(
        pd.to_datetime(date_values).unique()
    )

    df = df[
        df["Date"].isin(requested_dates)
    ].copy()

    df = df.set_index("Date")

    return df


# ---------------------------------------------------------------------
# 5. READ REBUILT EXTRACTED DATA
# ---------------------------------------------------------------------

def read_rebuilt_grid_file(model, variable):

    filepath = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
    )

    if not os.path.isfile(filepath):
        raise FileNotFoundError(filepath)

    df = pd.read_csv(filepath)

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="raise"
    )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates in rebuilt file: {filepath}"
        )

    return df.set_index("Date")


# ---------------------------------------------------------------------
# 6. SOURCE-LEVEL INVESTIGATION
# ---------------------------------------------------------------------

trace_records = []

models = sorted(
    violations["Model"].unique()
)

for model in models:

    print("\n" + "=" * 70)
    print("INVESTIGATING MODEL:", model)
    print("=" * 70)

    model_events = violations[
        violations["Model"] == model
    ].copy()

    grid_indices = sorted(
        model_events["Grid_Index"].unique()
    )

    required_dates = sorted(
        model_events["Date"].unique()
    )

    # Read rebuilt files
    rebuilt_tmax = read_rebuilt_grid_file(
        model, "TMaxData"
    )

    rebuilt_tmin = read_rebuilt_grid_file(
        model, "TMinData"
    )

    # Read original files, selecting only required columns
    original_tmax = read_selected_source(
        source_filepath(model, "TMaxData"),
        grid_indices,
        required_dates
    )

    original_tmin = read_selected_source(
        source_filepath(model, "TMinData"),
        grid_indices,
        required_dates
    )

    # Compare each reported violation
    for _, event in model_events.iterrows():

        date = event["Date"]
        grid_index = int(event["Grid_Index"])

        col = f"grid_col_{grid_index}"

        record = {
            "Model": model,
            "Date": date,
            "Grid_Index": grid_index,
            "Grid_Column": col,
            "Reported_Tmax_C": event["Tmax_C"],
            "Reported_Tmin_C": event["Tmin_C"],
            "Reported_Reversal_C": event[
                "Tmin_minus_Tmax_C"
            ]
        }

        # Retrieve original source values
        if (
            date in original_tmax.index
            and date in original_tmin.index
            and col in original_tmax.columns
            and col in original_tmin.columns
        ):

            raw_tmax = float(
                original_tmax.loc[date, col]
            )

            raw_tmin = float(
                original_tmin.loc[date, col]
            )

        else:
            raw_tmax = np.nan
            raw_tmin = np.nan

        # Retrieve rebuilt extracted values
        if (
            date in rebuilt_tmax.index
            and date in rebuilt_tmin.index
            and col in rebuilt_tmax.columns
            and col in rebuilt_tmin.columns
        ):

            extracted_tmax = float(
                rebuilt_tmax.loc[date, col]
            )

            extracted_tmin = float(
                rebuilt_tmin.loc[date, col]
            )

        else:
            extracted_tmax = np.nan
            extracted_tmin = np.nan

        # Test two possible unit interpretations:
        # 1. Source already in Celsius
        # 2. Source in Kelvin, converted to Celsius
        record.update({

            "Original_Tmax_Raw": raw_tmax,
            "Original_Tmin_Raw": raw_tmin,

            "Extracted_Tmax_C": extracted_tmax,
            "Extracted_Tmin_C": extracted_tmin,

            "Original_Raw_Tmax_Minus_Tmin": (
                raw_tmax - raw_tmin
                if np.isfinite(raw_tmax)
                and np.isfinite(raw_tmin)
                else np.nan
            ),

            "Raw_Tmax_Minus_273_15": (
                raw_tmax - 273.15
                if np.isfinite(raw_tmax)
                else np.nan
            ),

            "Raw_Tmin_Minus_273_15": (
                raw_tmin - 273.15
                if np.isfinite(raw_tmin)
                else np.nan
            ),

            "Tmax_Extraction_Difference_Celsius": (
                extracted_tmax - raw_tmax
                if np.isfinite(extracted_tmax)
                and np.isfinite(raw_tmax)
                else np.nan
            ),

            "Tmax_Extraction_Difference_KelvinConversion": (
                extracted_tmax - (raw_tmax - 273.15)
                if np.isfinite(extracted_tmax)
                and np.isfinite(raw_tmax)
                else np.nan
            ),

            "Tmin_Extraction_Difference_Celsius": (
                extracted_tmin - raw_tmin
                if np.isfinite(extracted_tmin)
                and np.isfinite(raw_tmin)
                else np.nan
            ),

            "Tmin_Extraction_Difference_KelvinConversion": (
                extracted_tmin - (raw_tmin - 273.15)
                if np.isfinite(extracted_tmin)
                and np.isfinite(raw_tmin)
                else np.nan
            )
        })

        record["Original_Source_Reversal"] = (
            raw_tmax < raw_tmin
            if np.isfinite(raw_tmax)
            and np.isfinite(raw_tmin)
            else False
        )

        record["Original_Tmax_Zero"] = (
            np.isclose(raw_tmax, 0)
            if np.isfinite(raw_tmax)
            else False
        )

        record["Original_Tmin_Zero"] = (
            np.isclose(raw_tmin, 0)
            if np.isfinite(raw_tmin)
            else False
        )

        record["Extracted_Tmax_Minus_273_15"] = (
            np.isclose(extracted_tmax, -273.15)
            if np.isfinite(extracted_tmax)
            else False
        )

        record["Extracted_Tmin_Minus_273_15"] = (
            np.isclose(extracted_tmin, -273.15)
            if np.isfinite(extracted_tmin)
            else False
        )

        # Classify the comparison, without correcting data.
        if not np.isfinite(raw_tmax) or not np.isfinite(raw_tmin):
            classification = "SOURCE_VALUE_MISSING"

        elif (
            np.isfinite(extracted_tmax)
            and np.isfinite(extracted_tmin)
            and np.isclose(raw_tmax, 0)
            and np.isclose(extracted_tmax, -273.15)
        ):
            classification = "ZERO_SOURCE_TO_MINUS_273_15"

        elif (
            np.isfinite(extracted_tmax)
            and np.isfinite(extracted_tmin)
            and np.isclose(
                extracted_tmax,
                raw_tmax - 273.15,
                atol=1e-5
            )
            and np.isclose(
                extracted_tmin,
                raw_tmin - 273.15,
                atol=1e-5
            )
        ):
            classification = "KELVIN_CONVERSION_MATCH"

        elif (
            np.isfinite(extracted_tmax)
            and np.isfinite(extracted_tmin)
            and np.isclose(
                extracted_tmax,
                raw_tmax,
                atol=1e-5
            )
            and np.isclose(
                extracted_tmin,
                raw_tmin,
                atol=1e-5
            )
        ):
            classification = "DIRECT_CELSIUS_MATCH"

        else:
            classification = "EXTRACTION_MISMATCH_OR_OTHER"

        record["Source_Trace_Classification"] = classification

        trace_records.append(record)

# ---------------------------------------------------------------------
# 7. SAVE DETAILED TRACE
# ---------------------------------------------------------------------

trace = pd.DataFrame(trace_records)

trace.to_csv(
    OUTPUT_FILE,
    index=False
)

# ---------------------------------------------------------------------
# 8. MODEL SUMMARY
# ---------------------------------------------------------------------

summary = (
    trace.groupby("Model")
    .agg(
        Total_Events=("Date", "count"),

        Original_Source_Reversals=(
            "Original_Source_Reversal", "sum"
        ),

        Original_Tmax_Zero_Count=(
            "Original_Tmax_Zero", "sum"
        ),

        Original_Tmin_Zero_Count=(
            "Original_Tmin_Zero", "sum"
        ),

        Extracted_Tmax_Minus_273_15_Count=(
            "Extracted_Tmax_Minus_273_15", "sum"
        ),

        Extracted_Tmin_Minus_273_15_Count=(
            "Extracted_Tmin_Minus_273_15", "sum"
        )
    )
    .reset_index()
)

classification_summary = pd.crosstab(
    trace["Model"],
    trace["Source_Trace_Classification"]
).reset_index()

summary = summary.merge(
    classification_summary,
    on="Model",
    how="left"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

# ---------------------------------------------------------------------
# 9. DISPLAY RESULTS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("ORIGINAL SOURCE TRACE: MODEL SUMMARY")
print("=" * 75)

display(summary)

print("\n" + "=" * 75)
print("LARGEST SOURCE-LEVEL VIOLATIONS")
print("=" * 75)

display(
    trace.sort_values(
        "Reported_Reversal_C",
        ascending=False
    ).head(25)
)

print("\n" + "=" * 75)
print("EXACT -273.15 DEGREE CASES")
print("=" * 75)

display(
    trace[
        trace["Extracted_Tmax_Minus_273_15"]
        | trace["Extracted_Tmin_Minus_273_15"]
    ].head(30)
)

print("\n" + "=" * 75)
print("OUTPUT FILES")
print("=" * 75)

print("Detailed report:")
print(OUTPUT_FILE)

print("\nModel summary:")
print(SUMMARY_FILE)

print("\nTotal events traced:", len(trace))

print("\nORIGINAL SOURCE INVESTIGATION COMPLETED.")


**[orig #225]** Full source-level temperature quality audit (five models with violations)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# =====================================================================
# FULL SOURCE-LEVEL TEMPERATURE QUALITY AUDIT
# Alwar | CMIP6 SSP245 | 2015-2100
#
# Five models with source-level Tmax < Tmin events.
# All 22 Alwar grid cells are investigated.
#
# No automatic corrections are applied.
# =====================================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------
# 1. PATHS AND SETTINGS
# ---------------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

SOURCE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

GRID_DIR = os.path.join(
    BASE_DIR, "Alwar_daily_extracted_REBUILT"
)

OUT_DIR = os.path.join(
    BASE_DIR, "Climate_Indicators"
)

os.makedirs(OUT_DIR, exist_ok=True)

MODELS = [
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "INM-CM4-8",
    "INM-CM5-0"
]

# Diagnostic thresholds, not automatic deletion rules
SUSPECT_LOW = -50.0
SUSPECT_HIGH = 60.0
SENTINEL_VALUE = -273.15
TOLERANCE = 1e-5

# ---------------------------------------------------------------------
# 2. SOURCE READER
# ---------------------------------------------------------------------

def read_source_selected(filepath, grid_indices):

    grid_indices = sorted(set(int(x) for x in grid_indices))

    usecols = (
        [0, 1, 2]
        + [3 + i for i in grid_indices]
    )

    names = (
        ["Year", "Month", "Day"]
        + [f"grid_col_{i}" for i in grid_indices]
    )

    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        skiprows=2,
        usecols=usecols,
        names=names,
        engine="c"
    )

    for col in names:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df["Date"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        ),
        errors="coerce"
    )

    if df["Date"].isna().any():
        raise ValueError(
            f"Invalid dates in {filepath}"
        )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates in {filepath}"
        )

    return df.set_index("Date").sort_index()


# ---------------------------------------------------------------------
# 3. REBUILT DATA READER
# ---------------------------------------------------------------------

def read_rebuilt(model, variable):

    filepath = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
    )

    if not os.path.isfile(filepath):
        raise FileNotFoundError(filepath)

    df = pd.read_csv(filepath)

    df["Date"] = pd.to_datetime(df["Date"])

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates in {filepath}"
        )

    return df.set_index("Date").sort_index()


# ---------------------------------------------------------------------
# 4. INITIALIZE OUTPUTS
# ---------------------------------------------------------------------

grid_summary_records = []
model_summary_records = []
monthly_records = []
yearly_records = []
event_records = []
comparison_records = []

# ---------------------------------------------------------------------
# 5. PROCESS EACH MODEL
# ---------------------------------------------------------------------

for model in MODELS:

    print("\n" + "=" * 75)
    print("FULL SOURCE AUDIT:", model)
    print("=" * 75)

    tmax_path = os.path.join(
        SOURCE_DIR, model, "ssp245", "TMaxData"
    )

    tmin_path = os.path.join(
        SOURCE_DIR, model, "ssp245", "TMinData"
    )

    if not os.path.isfile(tmax_path):
        raise FileNotFoundError(tmax_path)

    if not os.path.isfile(tmin_path):
        raise FileNotFoundError(tmin_path)

    # Read rebuilt datasets to identify the 22 required cells.
    rebuilt_tmax = read_rebuilt(model, "TMaxData")
    rebuilt_tmin = read_rebuilt(model, "TMinData")

    grid_cols = [
        c for c in rebuilt_tmax.columns
        if c.startswith("grid_col_")
    ]

    grid_indices = sorted(
        int(c.replace("grid_col_", ""))
        for c in grid_cols
    )

    if len(grid_indices) != 22:
        raise ValueError(
            f"{model}: expected 22 grid cells, "
            f"found {len(grid_indices)}"
        )

    # Read original source files
    source_tmax = read_source_selected(
        tmax_path, grid_indices
    )

    source_tmin = read_source_selected(
        tmin_path, grid_indices
    )

    # Align all four datasets by exact date.
    common_dates = (
        source_tmax.index
        .intersection(source_tmin.index)
        .intersection(rebuilt_tmax.index)
        .intersection(rebuilt_tmin.index)
    )

    if len(common_dates) == 0:
        raise ValueError(
            f"No common dates for {model}"
        )

    source_tmax = source_tmax.loc[common_dates]
    source_tmin = source_tmin.loc[common_dates]

    rebuilt_tmax = rebuilt_tmax.loc[common_dates]
    rebuilt_tmin = rebuilt_tmin.loc[common_dates]

    model_reversals = 0
    model_valid_pairs = 0
    model_sentinel_tmax = 0
    model_sentinel_tmin = 0
    model_extreme_tmax = 0
    model_extreme_tmin = 0
    model_extraction_mismatches = 0

    # -----------------------------------------------------------------
    # 6. GRID-CELL ANALYSIS
    # -----------------------------------------------------------------

    for grid_index in grid_indices:

        col = f"grid_col_{grid_index}"

        tx = pd.to_numeric(
            source_tmax[col],
            errors="coerce"
        )

        tn = pd.to_numeric(
            source_tmin[col],
            errors="coerce"
        )

        rtx = pd.to_numeric(
            rebuilt_tmax[col],
            errors="coerce"
        )

        rtn = pd.to_numeric(
            rebuilt_tmin[col],
            errors="coerce"
        )

        valid = (
            np.isfinite(tx)
            & np.isfinite(tn)
        )

        reversal = valid & (tx < tn)

        sentinel_tx = np.isclose(
            tx, SENTINEL_VALUE,
            atol=TOLERANCE
        ) & np.isfinite(tx)

        sentinel_tn = np.isclose(
            tn, SENTINEL_VALUE,
            atol=TOLERANCE
        ) & np.isfinite(tn)

        extreme_tx = (
            np.isfinite(tx)
            & ((tx < SUSPECT_LOW) | (tx > SUSPECT_HIGH))
        )

        extreme_tn = (
            np.isfinite(tn)
            & ((tn < SUSPECT_LOW) | (tn > SUSPECT_HIGH))
        )

        mismatch_tx = (
            np.isfinite(tx)
            & np.isfinite(rtx)
            & (np.abs(tx - rtx) > TOLERANCE)
        )

        mismatch_tn = (
            np.isfinite(tn)
            & np.isfinite(rtn)
            & (np.abs(tn - rtn) > TOLERANCE)
        )

        mismatch_count = int(
            mismatch_tx.sum() + mismatch_tn.sum()
        )

        valid_count = int(valid.sum())
        reversal_count = int(reversal.sum())

        model_valid_pairs += valid_count
        model_reversals += reversal_count

        model_sentinel_tmax += int(sentinel_tx.sum())
        model_sentinel_tmin += int(sentinel_tn.sum())

        model_extreme_tmax += int(extreme_tx.sum())
        model_extreme_tmin += int(extreme_tn.sum())

        model_extraction_mismatches += mismatch_count

        grid_summary_records.append({

            "Model": model,
            "Grid_Index": grid_index,
            "Grid_Column": col,

            "Valid_Day_Pairs": valid_count,

            "Tmax_Below_Tmin_Count": reversal_count,

            "Tmax_Below_Tmin_Percent": (
                100 * reversal_count / valid_count
                if valid_count else np.nan
            ),

            "Tmax_Minus_273_15_Count": int(
                sentinel_tx.sum()
            ),

            "Tmin_Minus_273_15_Count": int(
                sentinel_tn.sum()
            ),

            "Tmax_Extreme_Count": int(
                extreme_tx.sum()
            ),

            "Tmin_Extreme_Count": int(
                extreme_tn.sum()
            ),

            "Tmax_Min": tx.min(),
            "Tmax_Max": tx.max(),
            "Tmin_Min": tn.min(),
            "Tmin_Max": tn.max(),

            "Source_Extraction_Mismatch_Count":
                mismatch_count
        })

        # -------------------------------------------------------------
        # 7. SAVE EVERY REVERSAL EVENT
        # -------------------------------------------------------------

        reversal_dates = tx.index[reversal]

        for date in reversal_dates:

            event_records.append({

                "Model": model,
                "Date": date,
                "Year": date.year,
                "Month": date.month,
                "Grid_Index": grid_index,

                "Original_Tmax_C": float(tx.loc[date]),
                "Original_Tmin_C": float(tn.loc[date]),

                "Tmin_Minus_Tmax_C": float(
                    tn.loc[date] - tx.loc[date]
                )
            })

        # -------------------------------------------------------------
        # 8. MONTHLY DISTRIBUTION
        # -------------------------------------------------------------

        monthly_temp = pd.DataFrame({
            "Tmax": tx,
            "Tmin": tn,
            "Reversal": reversal.astype(int)
        })

        monthly_group = monthly_temp.groupby(
            monthly_temp.index.to_period("M")
        )

        for period, group in monthly_group:

            monthly_records.append({

                "Model": model,
                "Grid_Index": grid_index,
                "Year_Month": str(period),

                "Valid_Pairs": int(
                    group[["Tmax", "Tmin"]].notna().all(axis=1).sum()
                ),

                "Reversal_Count": int(
                    group["Reversal"].sum()
                ),

                "Tmax_Mean": group["Tmax"].mean(),
                "Tmin_Mean": group["Tmin"].mean(),

                "Tmax_Minus_273_15_Count": int(
                    np.isclose(
                        group["Tmax"],
                        SENTINEL_VALUE,
                        atol=TOLERANCE
                    ).sum()
                )
            })

        # -------------------------------------------------------------
        # 9. YEARLY DISTRIBUTION
        # -------------------------------------------------------------

        yearly_temp = pd.DataFrame({
            "Tmax": tx,
            "Tmin": tn,
            "Reversal": reversal.astype(int)
        })

        yearly_group = yearly_temp.groupby(
            yearly_temp.index.year
        )

        for year, group in yearly_group:

            yearly_records.append({

                "Model": model,
                "Grid_Index": grid_index,
                "Year": int(year),

                "Valid_Pairs": int(
                    group[["Tmax", "Tmin"]].notna().all(axis=1).sum()
                ),

                "Reversal_Count": int(
                    group["Reversal"].sum()
                ),

                "Reversal_Percent": (
                    100 * group["Reversal"].sum()
                    / max(
                        group[["Tmax", "Tmin"]].notna().all(axis=1).sum(),
                        1
                    )
                )
            })

    # -----------------------------------------------------------------
    # 10. MODEL SUMMARY
    # -----------------------------------------------------------------

    model_summary_records.append({

        "Model": model,
        "Common_Dates": len(common_dates),
        "Grid_Cells": len(grid_indices),

        "Valid_Grid_Day_Pairs": model_valid_pairs,

        "Tmax_Below_Tmin_Count": model_reversals,

        "Tmax_Below_Tmin_Percent": (
            100 * model_reversals / model_valid_pairs
            if model_valid_pairs else np.nan
        ),

        "Tmax_Minus_273_15_Count": model_sentinel_tmax,
        "Tmin_Minus_273_15_Count": model_sentinel_tmin,

        "Tmax_Extreme_Count": model_extreme_tmax,
        "Tmin_Extreme_Count": model_extreme_tmin,

        "Source_Extraction_Mismatches":
            model_extraction_mismatches
    })

    print("Common dates:", len(common_dates))
    print("Grid cells:", len(grid_indices))
    print("Temperature reversals:", model_reversals)
    print("Tmax -273.15 count:", model_sentinel_tmax)
    print("Tmin -273.15 count:", model_sentinel_tmin)
    print("Source/extraction mismatches:",
          model_extraction_mismatches)


# ---------------------------------------------------------------------
# 11. CREATE OUTPUT TABLES
# ---------------------------------------------------------------------

grid_summary = pd.DataFrame(grid_summary_records)
model_summary = pd.DataFrame(model_summary_records)
monthly_summary = pd.DataFrame(monthly_records)
yearly_summary = pd.DataFrame(yearly_records)
event_table = pd.DataFrame(event_records)

# ---------------------------------------------------------------------
# 12. SAVE CSV FILES
# ---------------------------------------------------------------------

outputs = {

    "Temperature_Source_Audit_Model_Summary.csv":
        model_summary,

    "Temperature_Source_Audit_Grid_Summary.csv":
        grid_summary,

    "Temperature_Source_Audit_Monthly.csv":
        monthly_summary,

    "Temperature_Source_Audit_Yearly.csv":
        yearly_summary,

    "Temperature_Source_Audit_All_Reversals.csv":
        event_table
}

for filename, dataframe in outputs.items():

    path = os.path.join(OUT_DIR, filename)

    dataframe.to_csv(
        path,
        index=False
    )

    print("\nSaved:", path)


# ---------------------------------------------------------------------
# 13. DISPLAY RESULTS
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("MODEL-LEVEL SOURCE AUDIT")
print("=" * 75)

display(model_summary)

print("\n" + "=" * 75)
print("GRID-CELL SUMMARY")
print("=" * 75)

display(
    grid_summary.sort_values(
        ["Model", "Tmax_Below_Tmin_Count"],
        ascending=[True, False]
    )
)

print("\n" + "=" * 75)
print("YEARS WITH MOST REVERSALS")
print("=" * 75)

if not yearly_summary.empty:

    display(
        yearly_summary.sort_values(
            "Reversal_Count",
            ascending=False
        ).head(30)
    )

print("\n" + "=" * 75)
print("MONTHS WITH MOST REVERSALS")
print("=" * 75)

if not monthly_summary.empty:

    display(
        monthly_summary.sort_values(
            "Reversal_Count",
            ascending=False
        ).head(30)
    )

print("\n" + "=" * 75)
print("AUDIT COMPLETE")
print("=" * 75)


**[orig #226]** Complete source-level temperature QC audit, all 13 models  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# COMPLETE ALL-13-MODEL SOURCE-LEVEL TEMPERATURE QC AUDIT
# Study area: Alwar district, Rajasthan
# Variables: Tmax and Tmin | Scenario: SSP245
# Grid cells: 22 | Period: 2015-01-01 to 2100-12-31
#
# This code:
# 1. Reads original source files for all 13 models.
# 2. Audits missing values, -273.15 candidates and extremes.
# 3. Counts Tmax < Tmin reversals, with and without sentinel pairs.
# 4. Compares source values against rebuilt extracted grid-cell data.
# 5. Saves model, grid-cell and event-level audit reports.
#
# No source data are modified or corrected.
# ================================================================

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. PATHS AND SETTINGS
# -----------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

SOURCE_DIR = BASE_DIR / "India"

EXTRACTED_DIR = (
    BASE_DIR / "Alwar_grid_analysis" /
    "Alwar_daily_extracted_REBUILT"
)

WEIGHT_FILE = (
    BASE_DIR / "Alwar_grid_analysis" /
    "Area_weighted_spatial_selection" /
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = (
    BASE_DIR / "Alwar_grid_analysis" /
    "Climate_Indicators"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

SCENARIO = "ssp245"

# Provisional diagnostic thresholds only.
# These are not automatic scientific exclusion limits.
EXTREME_LOW = -50.0
EXTREME_HIGH = 60.0

# Exact source value to investigate as a possible missing-value sentinel.
SENTINEL = -273.15

EXPECTED_GRID_CELLS = 22

# Numerical tolerance for source/extraction comparison.
COMPARE_ATOL = 1e-8

# -----------------------------
# 2. CHECK INPUTS
# -----------------------------

for path in [SOURCE_DIR, EXTRACTED_DIR, WEIGHT_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

weights = pd.read_csv(WEIGHT_FILE)

required_weight_columns = [
    "column_index_zero_based",
    "longitude",
    "latitude"
]

missing_columns = [
    c for c in required_weight_columns if c not in weights.columns
]

if missing_columns:
    raise ValueError(
        f"Weight file is missing columns: {missing_columns}"
    )

grid_info = (
    weights[
        ["column_index_zero_based", "longitude", "latitude"]
    ]
    .drop_duplicates()
    .sort_values("column_index_zero_based")
    .reset_index(drop=True)
)

grid_indices = (
    grid_info["column_index_zero_based"]
    .astype(int)
    .tolist()
)

if len(grid_indices) != EXPECTED_GRID_CELLS:
    raise ValueError(
        f"Expected {EXPECTED_GRID_CELLS} grid cells, "
        f"but found {len(grid_indices)}."
    )

grid_lookup = {
    int(row["column_index_zero_based"]): (
        float(row["longitude"]),
        float(row["latitude"])
    )
    for _, row in grid_info.iterrows()
}

print("Input checks passed.")
print("Models:", len(MODELS))
print("Alwar grid cells:", len(grid_indices))
print("Grid indices:", grid_indices)
print("Output directory:", OUTPUT_DIR)

# -----------------------------
# 3. READ ORIGINAL SOURCE FILE
# -----------------------------

def read_original_source(file_path, selected_grid_indices):
    """
    Original source structure:
      First two rows: longitude and latitude metadata
      Remaining rows: Year Month Day followed by grid values

    Source grid cell index i is stored in source column 3+i.
    """

    source_columns = [0, 1, 2] + [
        3 + int(i) for i in selected_grid_indices
    ]

    names = ["Year", "Month", "Day"] + [
        f"grid_{i}" for i in selected_grid_indices
    ]

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        skiprows=2,
        header=None,
        usecols=source_columns,
        names=names,
        engine="c",
        low_memory=False
    )

    for c in names:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Date"] = pd.to_datetime(
        {
            "year": df["Year"],
            "month": df["Month"],
            "day": df["Day"]
        },
        errors="coerce"
    )

    if df["Date"].isna().any():
        bad = int(df["Date"].isna().sum())
        raise ValueError(
            f"{file_path}: {bad} invalid dates found."
        )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"{file_path}: duplicate dates found."
        )

    return df.sort_values("Date").reset_index(drop=True)


# -----------------------------
# 4. READ REBUILT EXTRACTION
# -----------------------------

def read_rebuilt_extraction(file_path, selected_grid_indices):
    """
    Reads the rebuilt extracted CSV.

    It identifies grid columns using names such as:
      grid_col_123
      grid_123

    If those names are unavailable, it uses the 22 grid columns
    in their existing CSV order, after checking the column count.
    """

    df = pd.read_csv(file_path)

    if "Date" not in df.columns:
        raise ValueError(
            f"'Date' column missing from {file_path}"
        )

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    if df["Date"].isna().any():
        raise ValueError(
            f"Invalid dates found in rebuilt file: {file_path}"
        )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates found in rebuilt file: {file_path}"
        )

    # First try exact grid-index matching.
    detected = {}

    for col in df.columns:
        match = re.fullmatch(
            r"(?:grid_col_|grid_)(\d+)",
            str(col)
        )
        if match:
            detected[int(match.group(1))] = col

    if all(i in detected for i in selected_grid_indices):
        mapping = {
            i: detected[i] for i in selected_grid_indices
        }
        method = "GRID_INDEX_HEADER_MATCH"

    else:
        # Fallback: identify data columns by excluding date fields.
        excluded = {"Date", "Year", "Month", "Day"}
        data_columns = [
            c for c in df.columns if c not in excluded
        ]

        if len(data_columns) != len(selected_grid_indices):
            raise ValueError(
                f"Cannot identify the 22 grid columns in {file_path}. "
                f"Found {len(data_columns)} candidate data columns. "
                "Inspect the CSV headers before proceeding."
            )

        mapping = {
            idx: col
            for idx, col in zip(
                selected_grid_indices, data_columns
            )
        }
        method = "POSITIONAL_MATCH_CHECK_REQUIRED"

    for col in mapping.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return (
        df.sort_values("Date").reset_index(drop=True),
        mapping,
        method
    )


# -----------------------------
# 5. COMPARISON FUNCTION
# -----------------------------

def compare_source_and_extraction(
    source_df,
    extracted_df,
    source_prefix,
    extracted_mapping,
    selected_grid_indices
):
    """
    Compares values after matching dates.
    Reports date coverage and value mismatches.
    """

    merged = source_df[["Date"]].merge(
        extracted_df[["Date"]],
        on="Date",
        how="outer",
        indicator=True
    )

    source_only_dates = int(
        (merged["_merge"] == "left_only").sum()
    )
    extracted_only_dates = int(
        (merged["_merge"] == "right_only").sum()
    )

    common_dates = pd.Index(
        source_df["Date"]
    ).intersection(
        pd.Index(extracted_df["Date"])
    )

    src = (
        source_df.set_index("Date")
        .loc[common_dates]
    )

    ext = (
        extracted_df.set_index("Date")
        .loc[common_dates]
    )

    mismatch_count = 0
    compared_count = 0

    for idx in selected_grid_indices:
        source_col = f"grid_{idx}"
        extracted_col = extracted_mapping[idx]

        a = pd.to_numeric(
            src[source_col], errors="coerce"
        ).to_numpy(dtype=float)

        b = pd.to_numeric(
            ext[extracted_col], errors="coerce"
        ).to_numpy(dtype=float)

        same_missing = np.isnan(a) & np.isnan(b)
        both_finite = np.isfinite(a) & np.isfinite(b)

        equal_finite = np.zeros(len(a), dtype=bool)

        equal_finite[both_finite] = np.isclose(
            a[both_finite],
            b[both_finite],
            atol=COMPARE_ATOL,
            rtol=0
        )

        equal = same_missing | equal_finite

        mismatch_count += int((~equal).sum())
        compared_count += len(a)

    return {
        "source_only_dates": source_only_dates,
        "extracted_only_dates": extracted_only_dates,
        "common_dates": len(common_dates),
        "values_compared": compared_count,
        "value_mismatches": mismatch_count,
        "date_sets_match": (
            source_only_dates == 0 and
            extracted_only_dates == 0
        )
    }


# -----------------------------
# 6. MODEL-LEVEL AUDIT
# -----------------------------

model_results = []
grid_results = []
reversal_events = []
extraction_results = []

for model_number, model in enumerate(MODELS, start=1):

    print("\n" + "=" * 70)
    print(f"MODEL {model_number}/{len(MODELS)}: {model}")
    print("=" * 70)

    tmax_path = (
        SOURCE_DIR / model / SCENARIO / "TMaxData"
    )
    tmin_path = (
        SOURCE_DIR / model / SCENARIO / "TMinData"
    )

    tmax_extract_path = (
        EXTRACTED_DIR /
        f"{model}_ssp245_TMaxData_Alwar_REBUILT.csv"
    )
    tmin_extract_path = (
        EXTRACTED_DIR /
        f"{model}_ssp245_TMinData_Alwar_REBUILT.csv"
    )

    required_files = [
        tmax_path,
        tmin_path,
        tmax_extract_path,
        tmin_extract_path
    ]

    missing_files = [
        str(p) for p in required_files if not p.exists()
    ]

    if missing_files:
        raise FileNotFoundError(
            "Missing files for model " + model + ":\n" +
            "\n".join(missing_files)
        )

    # Read original source files.
    tmax = read_original_source(
        tmax_path, grid_indices
    )
    tmin = read_original_source(
        tmin_path, grid_indices
    )

    # Align Tmax and Tmin by date.
    climate = tmax[["Date"]].merge(
        tmin[["Date"]],
        on="Date",
        how="outer",
        indicator=True
    )

    source_only_tmax_dates = int(
        (climate["_merge"] == "left_only").sum()
    )
    source_only_tmin_dates = int(
        (climate["_merge"] == "right_only").sum()
    )

    common_dates = pd.Index(
        tmax["Date"]
    ).intersection(
        pd.Index(tmin["Date"])
    )

    tx = (
        tmax.set_index("Date")
        .loc[common_dates]
    )
    tn = (
        tmin.set_index("Date")
        .loc[common_dates]
    )

    # Read rebuilt extracted data.
    tmax_ext, tmax_mapping, tmax_mapping_method = (
        read_rebuilt_extraction(
            tmax_extract_path, grid_indices
        )
    )

    tmin_ext, tmin_mapping, tmin_mapping_method = (
        read_rebuilt_extraction(
            tmin_extract_path, grid_indices
        )
    )

    # Source-to-extraction comparison.
    tmax_compare = compare_source_and_extraction(
        tmax,
        tmax_ext,
        "Tmax",
        tmax_mapping,
        grid_indices
    )

    tmin_compare = compare_source_and_extraction(
        tmin,
        tmin_ext,
        "Tmin",
        tmin_mapping,
        grid_indices
    )

    extraction_results.append({
        "model": model,
        "Tmax_mapping_method": tmax_mapping_method,
        "Tmin_mapping_method": tmin_mapping_method,
        "Tmax_source_only_dates": tmax_compare[
            "source_only_dates"
        ],
        "Tmax_extracted_only_dates": tmax_compare[
            "extracted_only_dates"
        ],
        "Tmax_values_compared": tmax_compare[
            "values_compared"
        ],
        "Tmax_value_mismatches": tmax_compare[
            "value_mismatches"
        ],
        "Tmin_source_only_dates": tmin_compare[
            "source_only_dates"
        ],
        "Tmin_extracted_only_dates": tmin_compare[
            "extracted_only_dates"
        ],
        "Tmin_values_compared": tmin_compare[
            "values_compared"
        ],
        "Tmin_value_mismatches": tmin_compare[
            "value_mismatches"
        ]
    })

    # Model-level counters.
    total_pairs = 0
    finite_pairs = 0
    sentinel_pairs = 0
    all_reversals = 0
    clean_reversals = 0
    tmax_missing = 0
    tmin_missing = 0
    tmax_sentinel = 0
    tmin_sentinel = 0
    tmax_extreme = 0
    tmin_extreme = 0
    tmax_non_sentinel_extreme = 0
    tmin_non_sentinel_extreme = 0

    for idx in grid_indices:

        tx_values = pd.to_numeric(
            tx[f"grid_{idx}"], errors="coerce"
        ).to_numpy(dtype=float)

        tn_values = pd.to_numeric(
            tn[f"grid_{idx}"], errors="coerce"
        ).to_numpy(dtype=float)

        dates = common_dates

        finite_tx = np.isfinite(tx_values)
        finite_tn = np.isfinite(tn_values)

        finite_pair = finite_tx & finite_tn

        tx_sentinel = (
            finite_tx &
            np.isclose(tx_values, SENTINEL, atol=1e-8, rtol=0)
        )
        tn_sentinel = (
            finite_tn &
            np.isclose(tn_values, SENTINEL, atol=1e-8, rtol=0)
        )

        either_sentinel = tx_sentinel | tn_sentinel

        # Reversals before excluding sentinel candidates.
        reversal = (
            finite_pair &
            (tx_values < tn_values)
        )

        # Reversals after provisionally excluding sentinel pairs.
        clean_pair = finite_pair & (~either_sentinel)

        clean_reversal = (
            clean_pair &
            (tx_values < tn_values)
        )

        # Provisional extremes.
        tx_extreme = (
            finite_tx &
            (
                (tx_values < EXTREME_LOW) |
                (tx_values > EXTREME_HIGH)
            )
        )

        tn_extreme = (
            finite_tn &
            (
                (tn_values < EXTREME_LOW) |
                (tn_values > EXTREME_HIGH)
            )
        )

        tx_non_sentinel_extreme = (
            tx_extreme & (~tx_sentinel)
        )
        tn_non_sentinel_extreme = (
            tn_extreme & (~tn_sentinel)
        )

        # Counts.
        total_pairs += len(dates)
        finite_pairs += int(finite_pair.sum())
        sentinel_pairs += int(
            (finite_pair & either_sentinel).sum()
        )

        all_reversals += int(reversal.sum())
        clean_reversals += int(clean_reversal.sum())

        tmax_missing += int((~finite_tx).sum())
        tmin_missing += int((~finite_tn).sum())

        tmax_sentinel += int(tx_sentinel.sum())
        tmin_sentinel += int(tn_sentinel.sum())

        tmax_extreme += int(tx_extreme.sum())
        tmin_extreme += int(tn_extreme.sum())

        tmax_non_sentinel_extreme += int(
            tx_non_sentinel_extreme.sum()
        )
        tmin_non_sentinel_extreme += int(
            tn_non_sentinel_extreme.sum()
        )

        lon, lat = grid_lookup[idx]

        # Grid-cell summary.
        grid_results.append({
            "model": model,
            "grid_index": idx,
            "longitude": lon,
            "latitude": lat,
            "common_dates": len(dates),
            "Tmax_nonfinite_count": int((~finite_tx).sum()),
            "Tmin_nonfinite_count": int((~finite_tn).sum()),
            "Tmax_minus273_15_count": int(tx_sentinel.sum()),
            "Tmin_minus273_15_count": int(tn_sentinel.sum()),
            "finite_temperature_pairs": int(finite_pair.sum()),
            "pairs_containing_sentinel": int(
                (finite_pair & either_sentinel).sum()
            ),
            "Tmax_extreme_count": int(tx_extreme.sum()),
            "Tmin_extreme_count": int(tn_extreme.sum()),
            "Tmax_non_sentinel_extreme_count": int(
                tx_non_sentinel_extreme.sum()
            ),
            "Tmin_non_sentinel_extreme_count": int(
                tn_non_sentinel_extreme.sum()
            ),
            "Tmax_less_than_Tmin_all": int(reversal.sum()),
            "Tmax_less_than_Tmin_non_sentinel": int(
                clean_reversal.sum()
            )
        })

        # Store each reversal event for detailed inspection.
        event_positions = np.where(reversal)[0]

        for pos in event_positions:
            reversal_events.append({
                "model": model,
                "Date": pd.Timestamp(dates[pos]),
                "grid_index": idx,
                "longitude": lon,
                "latitude": lat,
                "Tmax_C": float(tx_values[pos]),
                "Tmin_C": float(tn_values[pos]),
                "difference_Tmax_minus_Tmin_C": float(
                    tx_values[pos] - tn_values[pos]
                ),
                "Tmax_is_minus273_15": bool(
                    tx_sentinel[pos]
                ),
                "Tmin_is_minus273_15": bool(
                    tn_sentinel[pos]
                ),
                "contains_sentinel_candidate": bool(
                    either_sentinel[pos]
                ),
                "non_sentinel_reversal": bool(
                    clean_reversal[pos]
                )
            })

    # Model summary.
    model_results.append({
        "model": model,
        "Tmax_source_dates": len(tmax),
        "Tmin_source_dates": len(tmin),
        "common_Tmax_Tmin_dates": len(common_dates),
        "Tmax_only_dates": source_only_tmax_dates,
        "Tmin_only_dates": source_only_tmin_dates,
        "grid_cells_audited": len(grid_indices),
        "grid_day_pairs_expected": total_pairs,
        "finite_Tmax_Tmin_pairs": finite_pairs,
        "pairs_containing_sentinel_candidate": sentinel_pairs,
        "Tmax_nonfinite_count": tmax_missing,
        "Tmin_nonfinite_count": tmin_missing,
        "Tmax_minus273_15_count": tmax_sentinel,
        "Tmin_minus273_15_count": tmin_sentinel,
        "Tmax_extreme_count_screening": tmax_extreme,
        "Tmin_extreme_count_screening": tmin_extreme,
        "Tmax_non_sentinel_extreme_count": (
            tmax_non_sentinel_extreme
        ),
        "Tmin_non_sentinel_extreme_count": (
            tmin_non_sentinel_extreme
        ),
        "Tmax_less_than_Tmin_all": all_reversals,
        "Tmax_less_than_Tmin_non_sentinel": clean_reversals,
        "all_reversal_rate_percent_of_finite_pairs": (
            100 * all_reversals / finite_pairs
            if finite_pairs else np.nan
        ),
        "non_sentinel_reversal_rate_percent_of_clean_pairs": (
            100 * clean_reversals /
            (finite_pairs - sentinel_pairs)
            if finite_pairs > sentinel_pairs else np.nan
        ),
        "Tmax_source_extraction_mismatches": (
            tmax_compare["value_mismatches"]
        ),
        "Tmin_source_extraction_mismatches": (
            tmin_compare["value_mismatches"]
        ),
        "Tmax_source_extraction_date_sets_match": (
            tmax_compare["date_sets_match"]
        ),
        "Tmin_source_extraction_date_sets_match": (
            tmin_compare["date_sets_match"]
        )
    })

    print("Common Tmax/Tmin dates:", len(common_dates))
    print("Finite temperature pairs:", finite_pairs)
    print("Tmax -273.15 candidates:", tmax_sentinel)
    print("Tmin -273.15 candidates:", tmin_sentinel)
    print("All Tmax<Tmin reversals:", all_reversals)
    print("Non-sentinel reversals:", clean_reversals)
    print(
        "Source/extraction mismatches:",
        tmax_compare["value_mismatches"],
        "(Tmax),",
        tmin_compare["value_mismatches"],
        "(Tmin)"
    )

# -----------------------------
# 7. CREATE REPORT DATAFRAMES
# -----------------------------

model_summary = pd.DataFrame(model_results)
grid_summary = pd.DataFrame(grid_results)
event_summary = pd.DataFrame(reversal_events)
extraction_summary = pd.DataFrame(extraction_results)

if not event_summary.empty:
    event_summary = event_summary.sort_values(
        ["model", "Date", "grid_index"]
    )

# -----------------------------
# 8. SAVE REPORTS
# -----------------------------

model_file = (
    OUTPUT_DIR / "All13_Temperature_Source_Audit_Model_Summary.csv"
)

grid_file = (
    OUTPUT_DIR / "All13_Temperature_Source_Audit_Grid_Summary.csv"
)

event_file = (
    OUTPUT_DIR / "All13_Temperature_Source_Audit_All_Reversals.csv"
)

extraction_file = (
    OUTPUT_DIR / "All13_Temperature_Source_Audit_Extraction_Check.csv"
)

model_summary.to_csv(model_file, index=False)
grid_summary.to_csv(grid_file, index=False)
event_summary.to_csv(event_file, index=False)
extraction_summary.to_csv(extraction_file, index=False)

# -----------------------------
# 9. FINAL AUDIT CHECKS
# -----------------------------

print("\n" + "=" * 70)
print("ALL-13-MODEL AUDIT COMPLETED")
print("=" * 70)

print("\nMODEL SUMMARY")
display(model_summary)

print("\nEXTRACTION INTEGRITY SUMMARY")
display(extraction_summary)

print("\nGRID-CELL SUMMARY: first 15 rows")
display(grid_summary.head(15))

print("\nREVERSAL EVENTS: first 20 rows")
display(event_summary.head(20))

print("\nTOTALS ACROSS ALL MODELS")
print("Models audited:", model_summary["model"].nunique())
print(
    "Grid-cell model combinations:",
    len(grid_summary)
)
print(
    "Total Tmax<Tmin reversals:",
    int(model_summary["Tmax_less_than_Tmin_all"].sum())
)
print(
    "Total non-sentinel reversals:",
    int(
        model_summary[
            "Tmax_less_than_Tmin_non_sentinel"
        ].sum()
    )
)
print(
    "Total Tmax -273.15 candidates:",
    int(model_summary["Tmax_minus273_15_count"].sum())
)
print(
    "Total Tmin -273.15 candidates:",
    int(model_summary["Tmin_minus273_15_count"].sum())
)
print(
    "Total Tmax non-finite values:",
    int(model_summary["Tmax_nonfinite_count"].sum())
)
print(
    "Total Tmin non-finite values:",
    int(model_summary["Tmin_nonfinite_count"].sum())
)
print(
    "Total source/extraction mismatches:",
    int(
        extraction_summary[
            "Tmax_value_mismatches"
        ].sum()
        +
        extraction_summary[
            "Tmin_value_mismatches"
        ].sum()
    )
)

print("\nSAVED FILES:")
for f in [
    model_file,
    grid_file,
    event_file,
    extraction_file
]:
    print(f)

print("\nIMPORTANT:")
print(
    "- -273.15°C is treated as a candidate sentinel, "
    "not a confirmed missing-value code."
)
print(
    "- Extreme thresholds are provisional screening flags."
)
print(
    "- No temperature values were corrected, removed, "
    "swapped or interpolated."
)


## 3. Where and when the reversals occur

Some cells here need variables from the cell directly above them; the labels say which.

**[orig #227]** Map the frequency of Tmax < Tmin reversals by grid cell  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# SPATIAL MAPPING OF Tmax < Tmin REVERSAL FREQUENCY
# Alwar District, Rajasthan | CMIP6 SSP245
#
# Models:
#   CanESM5
#   INM-CM4-8
#   INM-CM5-0
#
# Input:
#   All13_Temperature_Source_Audit_Grid_Summary.csv
#   LGD_Blocks.geojson
#
# Outputs:
#   1. Three-panel spatial reversal frequency map
#   2. Individual model maps
#   3. Grid-cell reversal frequency CSV
#
# No raw climate data are modified.
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------
# 1. PATHS
# -----------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/gw_framework"
)

AUDIT_DIR = (
    BASE_DIR /
    "climate_forcing" /
    "Vimal_Mishra_CMIP6" /
    "SSP245_extracted" /
    "Alwar_grid_analysis" /
    "Climate_Indicators"
)

GRID_AUDIT_FILE = (
    AUDIT_DIR /
    "All13_Temperature_Source_Audit_Grid_Summary.csv"
)

BLOCK_GEOJSON = (
    BASE_DIR /
    "Shapefile" /
    "LGD_Blocks.geojson"
)

OUTPUT_DIR = AUDIT_DIR / "Reversal_Spatial_Maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2. MODELS TO MAP
# -----------------------------

MODELS = [
    "CanESM5",
    "INM-CM4-8",
    "INM-CM5-0"
]

# -----------------------------
# 3. CHECK FILES
# -----------------------------

for f in [GRID_AUDIT_FILE, BLOCK_GEOJSON]:
    if not f.exists():
        raise FileNotFoundError(f"File not found: {f}")

print("Input files found.")

# -----------------------------
# 4. READ GRID AUDIT
# -----------------------------

grid_df = pd.read_csv(GRID_AUDIT_FILE)

required_columns = [
    "model",
    "grid_index",
    "longitude",
    "latitude",
    "common_dates",
    "Tmax_less_than_Tmin_non_sentinel"
]

missing = [
    c for c in required_columns
    if c not in grid_df.columns
]

if missing:
    raise ValueError(
        f"Missing columns in grid audit file: {missing}"
    )

# Select affected models.
map_df = grid_df[
    grid_df["model"].isin(MODELS)
].copy()

if map_df.empty:
    raise ValueError("No records found for selected models.")

# Ensure numeric data.
for col in [
    "grid_index",
    "longitude",
    "latitude",
    "common_dates",
    "Tmax_less_than_Tmin_non_sentinel"
]:
    map_df[col] = pd.to_numeric(
        map_df[col], errors="coerce"
    )

if map_df[
    required_columns[1:]
].isna().any().any():
    raise ValueError(
        "Missing or non-numeric grid audit values found."
    )

# Check grid-cell coverage.
for model in MODELS:
    n = map_df.loc[
        map_df["model"] == model,
        "grid_index"
    ].nunique()

    if n != 22:
        raise ValueError(
            f"{model}: expected 22 unique grid cells, found {n}."
        )

# -----------------------------
# 5. CALCULATE FREQUENCY
# -----------------------------

map_df["reversal_count"] = (
    map_df["Tmax_less_than_Tmin_non_sentinel"]
)

map_df["reversal_frequency_percent"] = (
    100.0 *
    map_df["reversal_count"] /
    map_df["common_dates"]
)

# Frequency per 10,000 days, useful for interpretation.
map_df["reversals_per_10000_days"] = (
    10000.0 *
    map_df["reversal_count"] /
    map_df["common_dates"]
)

map_df["QC_flag"] = np.where(
    map_df["reversal_count"] > 0,
    "REVERSAL_PRESENT",
    "NO_REVERSAL"
)

# Save detailed grid-cell table.
map_csv = OUTPUT_DIR / (
    "Alwar_NonSentinel_Temperature_Reversal_Frequency.csv"
)

map_df.to_csv(map_csv, index=False)

print("\nGrid-cell reversal frequency table:")
display(
    map_df[
        [
            "model",
            "grid_index",
            "longitude",
            "latitude",
            "reversal_count",
            "common_dates",
            "reversal_frequency_percent",
            "QC_flag"
        ]
    ].sort_values(
        ["model", "reversal_count"],
        ascending=[True, False]
    )
)

# -----------------------------
# 6. READ BLOCK BOUNDARIES
# -----------------------------

try:
    import geopandas as gpd
except ImportError:
    raise ImportError(
        "GeoPandas is not installed. Run: "
        "!pip install geopandas"
    )

blocks = gpd.read_file(BLOCK_GEOJSON)

if blocks.crs is None:
    raise ValueError(
        "Block GeoJSON has no CRS. Define its CRS before mapping."
    )

blocks = blocks.to_crs("EPSG:4326")

# -----------------------------
# 7. MAP SETTINGS
# -----------------------------

# Use one common colour scale across all three models.
vmax = max(
    float(map_df["reversal_frequency_percent"].max()),
    0.001
)

# -----------------------------
# 8. THREE-PANEL COMPARISON MAP
# -----------------------------

fig, axes = plt.subplots(
    1, 3,
    figsize=(19, 8),
    constrained_layout=True
)

for ax, model in zip(axes, MODELS):

    subset = map_df[
        map_df["model"] == model
    ].copy()

    # Plot block boundaries.
    blocks.boundary.plot(
        ax=ax,
        linewidth=0.8
    )

    # Plot all 22 grid cells.
    scatter = ax.scatter(
        subset["longitude"],
        subset["latitude"],
        c=subset["reversal_frequency_percent"],
        s=145,
        marker="s",
        vmin=0,
        vmax=vmax,
        edgecolors="black",
        linewidths=0.7
    )

    # Add grid-cell index labels.
    for _, row in subset.iterrows():
        ax.annotate(
            str(int(row["grid_index"])),
            (
                row["longitude"],
                row["latitude"]
            ),
            xytext=(0, 0),
            textcoords="offset points",
            ha="center",
            va="center",
            fontsize=6
        )

    total_reversals = int(
        subset["reversal_count"].sum()
    )

    ax.set_title(
        f"{model}\n"
        f"Total non-sentinel reversals: {total_reversals}",
        fontsize=11
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

cbar = fig.colorbar(
    scatter,
    ax=axes,
    shrink=0.78,
    pad=0.02
)

cbar.set_label(
    "Non-sentinel Tmax < Tmin frequency (%)",
    fontsize=10
)

fig.suptitle(
    "Spatial Distribution of Temperature Reversal Frequency\n"
    "Alwar District, Rajasthan | CMIP6 SSP245",
    fontsize=15,
    fontweight="bold"
)

comparison_png = (
    OUTPUT_DIR /
    "Alwar_Temperature_Reversal_Three_Model_Comparison.png"
)

plt.savefig(
    comparison_png,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# 9. INDIVIDUAL MODEL MAPS
# -----------------------------

for model in MODELS:

    subset = map_df[
        map_df["model"] == model
    ].copy()

    fig, ax = plt.subplots(
        figsize=(9, 9)
    )

    blocks.boundary.plot(
        ax=ax,
        linewidth=1.0
    )

    scatter = ax.scatter(
        subset["longitude"],
        subset["latitude"],
        c=subset["reversal_frequency_percent"],
        s=190,
        marker="s",
        vmin=0,
        vmax=vmax,
        edgecolors="black",
        linewidths=0.8
    )

    for _, row in subset.iterrows():
        ax.annotate(
            (
                f'{int(row["grid_index"])}\n'
                f'{int(row["reversal_count"])}'
            ),
            (
                row["longitude"],
                row["latitude"]
            ),
            ha="center",
            va="center",
            fontsize=7
        )

    cbar = plt.colorbar(
        scatter,
        ax=ax,
        shrink=0.75
    )

    cbar.set_label(
        "Reversal frequency (%)"
    )

    ax.set_title(
        f"{model}: Alwar Grid-Cell Reversal Frequency\n"
        "Non-sentinel Tmax < Tmin observations",
        fontsize=12
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    plt.tight_layout()

    individual_png = (
        OUTPUT_DIR /
        f"Alwar_Reversal_Map_{model}.png"
    )

    plt.savefig(
        individual_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# -----------------------------
# 10. MODEL SUMMARY
# -----------------------------

summary = (
    map_df.groupby("model")
    .agg(
        grid_cells=("grid_index", "nunique"),
        total_reversals=("reversal_count", "sum"),
        affected_grid_cells=(
            "reversal_count",
            lambda x: int((x > 0).sum())
        ),
        maximum_cell_reversals=(
            "reversal_count", "max"
        ),
        mean_cell_frequency_percent=(
            "reversal_frequency_percent", "mean"
        ),
        maximum_cell_frequency_percent=(
            "reversal_frequency_percent", "max"
        )
    )
    .reset_index()
)

summary_file = (
    OUTPUT_DIR /
    "Alwar_Reversal_Model_Spatial_Summary.csv"
)

summary.to_csv(
    summary_file,
    index=False
)

print("\nMODEL SPATIAL SUMMARY")
display(summary)

# -----------------------------
# 11. OUTPUT INVENTORY
# -----------------------------

print("\nSPATIAL MAPPING COMPLETED")
print("Output folder:", OUTPUT_DIR)

print("\nFiles created:")
print(map_csv)
print(comparison_png)
print(summary_file)

for model in MODELS:
    print(
        OUTPUT_DIR /
        f"Alwar_Reversal_Map_{model}.png"
    )


**[orig #228]** Temporal and magnitude QC of the reversals  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# TEMPORAL AND MAGNITUDE QC OF Tmax < Tmin REVERSALS
# Alwar, Rajasthan | CMIP6 SSP245
#
# Models:
#   CanESM5
#   INM-CM4-8
#   INM-CM5-0
#
# Analyses:
#   1. Seasonal reversal frequency
#   2. Annual reversal frequency
#   3. Monthly reversal frequency
#   4. Reversal magnitude distribution
#   5. Event-level date and grid-cell inventory
#
# Uses original source files. Does not alter climate data.
# ================================================================

# -----------------------------
# 1. MOUNT GOOGLE DRIVE
# -----------------------------

from google.colab import drive
drive.mount("/content/drive")

# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------
# 3. DEFINE PATHS
# -----------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/SSP245_extracted"
)

SOURCE_DIR = BASE_DIR / "India"

WEIGHT_FILE = (
    BASE_DIR /
    "Alwar_grid_analysis" /
    "Area_weighted_spatial_selection" /
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = (
    BASE_DIR /
    "Alwar_grid_analysis" /
    "Climate_Indicators" /
    "Reversal_Temporal_Magnitude_QC"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    "CanESM5",
    "INM-CM4-8",
    "INM-CM5-0"
]

SCENARIO = "ssp245"
SENTINEL = -273.15

# -----------------------------
# 4. CHECK INPUTS
# -----------------------------

if not SOURCE_DIR.exists():
    raise FileNotFoundError(SOURCE_DIR)

if not WEIGHT_FILE.exists():
    raise FileNotFoundError(WEIGHT_FILE)

weights = pd.read_csv(WEIGHT_FILE)

grid_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

if len(grid_indices) != 22:
    raise ValueError(
        f"Expected 22 grid cells; found {len(grid_indices)}"
    )

print("Drive mounted and input files found.")
print("Models:", MODELS)
print("Grid cells:", len(grid_indices))

# -----------------------------
# 5. READ ORIGINAL SOURCE FILES
# -----------------------------

def read_source(filepath, grid_indices):

    usecols = (
        [0, 1, 2] +
        [3 + int(i) for i in grid_indices]
    )

    names = (
        ["Year", "Month", "Day"] +
        [f"grid_{i}" for i in grid_indices]
    )

    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        skiprows=2,
        header=None,
        usecols=usecols,
        names=names,
        engine="c",
        low_memory=False
    )

    for col in names:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

    df["Date"] = pd.to_datetime(
        {
            "year": df["Year"],
            "month": df["Month"],
            "day": df["Day"]
        },
        errors="coerce"
    )

    if df["Date"].isna().any():
        raise ValueError(
            f"Invalid dates in {filepath}"
        )

    if df["Date"].duplicated().any():
        raise ValueError(
            f"Duplicate dates in {filepath}"
        )

    return df.sort_values("Date").reset_index(drop=True)


# -----------------------------
# 6. SEASON CLASSIFICATION
# -----------------------------

def assign_season(month):

    if month in [1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Pre-monsoon"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    elif month in [10, 11, 12]:
        return "Post-monsoon"

    return "Unknown"


SEASON_ORDER = [
    "Winter",
    "Pre-monsoon",
    "Monsoon",
    "Post-monsoon"
]

# -----------------------------
# 7. PROCESS ALL THREE MODELS
# -----------------------------

event_records = []
daily_records = []

for model in MODELS:

    print("\nProcessing:", model)

    tmax_path = (
        SOURCE_DIR / model / SCENARIO / "TMaxData"
    )

    tmin_path = (
        SOURCE_DIR / model / SCENARIO / "TMinData"
    )

    if not tmax_path.exists() or not tmin_path.exists():
        raise FileNotFoundError(
            f"Missing Tmax or Tmin source for {model}"
        )

    tmax = read_source(tmax_path, grid_indices)
    tmin = read_source(tmin_path, grid_indices)

    common_dates = (
        pd.Index(tmax["Date"])
        .intersection(pd.Index(tmin["Date"]))
        .sort_values()
    )

    tx = tmax.set_index("Date").loc[common_dates]
    tn = tmin.set_index("Date").loc[common_dates]

    # One daily record per date and grid cell.
    for grid_idx in grid_indices:

        tx_values = tx[f"grid_{grid_idx}"].to_numpy(
            dtype=float
        )

        tn_values = tn[f"grid_{grid_idx}"].to_numpy(
            dtype=float
        )

        finite_pair = (
            np.isfinite(tx_values) &
            np.isfinite(tn_values)
        )

        tx_sentinel = (
            np.isclose(
                tx_values,
                SENTINEL,
                atol=1e-8,
                rtol=0
            )
        )

        tn_sentinel = (
            np.isclose(
                tn_values,
                SENTINEL,
                atol=1e-8,
                rtol=0
            )
        )

        # Exclude any pair containing a candidate sentinel.
        clean_pair = (
            finite_pair &
            (~tx_sentinel) &
            (~tn_sentinel)
        )

        reversal = (
            clean_pair &
            (tx_values < tn_values)
        )

        for pos, date in enumerate(common_dates):

            if not clean_pair[pos]:
                continue

            difference = (
                tx_values[pos] - tn_values[pos]
            )

            daily_records.append({
                "model": model,
                "Date": date,
                "year": date.year,
                "month": date.month,
                "season": assign_season(date.month),
                "grid_index": grid_idx,
                "Tmax_C": tx_values[pos],
                "Tmin_C": tn_values[pos],
                "difference_C": difference,
                "is_reversal": bool(reversal[pos])
            })

            if reversal[pos]:

                event_records.append({
                    "model": model,
                    "Date": date,
                    "year": date.year,
                    "month": date.month,
                    "season": assign_season(date.month),
                    "grid_index": grid_idx,
                    "Tmax_C": tx_values[pos],
                    "Tmin_C": tn_values[pos],
                    "difference_C": difference,
                    "magnitude_C": abs(difference)
                })

    print(
        "Common dates:", len(common_dates),
        "| Reversal events:",
        sum(
            1 for e in event_records
            if e["model"] == model
        )
    )

# -----------------------------
# 8. CREATE DATAFRAMES
# -----------------------------

daily_df = pd.DataFrame(daily_records)
events_df = pd.DataFrame(event_records)

if daily_df.empty:
    raise ValueError("No valid daily pairs were found.")

if events_df.empty:
    raise ValueError("No non-sentinel reversals were found.")

daily_df["Date"] = pd.to_datetime(daily_df["Date"])
events_df["Date"] = pd.to_datetime(events_df["Date"])

print("\nTotal valid daily grid-cell pairs:", len(daily_df))
print("Total non-sentinel reversal events:", len(events_df))

# -----------------------------
# 9. SEASONAL FREQUENCY
# -----------------------------

seasonal = (
    daily_df.groupby(
        ["model", "season"],
        observed=True
    )
    .agg(
        valid_pairs=("is_reversal", "size"),
        reversal_count=("is_reversal", "sum")
    )
    .reset_index()
)

seasonal["frequency_percent"] = (
    100 *
    seasonal["reversal_count"] /
    seasonal["valid_pairs"]
)

seasonal["season"] = pd.Categorical(
    seasonal["season"],
    categories=SEASON_ORDER,
    ordered=True
)

seasonal = seasonal.sort_values(
    ["model", "season"]
)

seasonal_file = (
    OUTPUT_DIR / "Seasonal_Reversal_Frequency.csv"
)

seasonal.to_csv(seasonal_file, index=False)

print("\nSEASONAL REVERSAL FREQUENCY")
display(seasonal)

# -----------------------------
# 10. ANNUAL FREQUENCY
# -----------------------------

annual = (
    daily_df.groupby(
        ["model", "year"],
        observed=True
    )
    .agg(
        valid_pairs=("is_reversal", "size"),
        reversal_count=("is_reversal", "sum")
    )
    .reset_index()
)

annual["frequency_percent"] = (
    100 *
    annual["reversal_count"] /
    annual["valid_pairs"]
)

annual_file = (
    OUTPUT_DIR / "Annual_Reversal_Frequency.csv"
)

annual.to_csv(annual_file, index=False)

# -----------------------------
# 11. MONTHLY FREQUENCY
# -----------------------------

monthly = (
    daily_df.groupby(
        ["model", "month"],
        observed=True
    )
    .agg(
        valid_pairs=("is_reversal", "size"),
        reversal_count=("is_reversal", "sum")
    )
    .reset_index()
)

monthly["frequency_percent"] = (
    100 *
    monthly["reversal_count"] /
    monthly["valid_pairs"]
)

monthly_file = (
    OUTPUT_DIR / "Monthly_Reversal_Frequency.csv"
)

monthly.to_csv(monthly_file, index=False)

print("\nMONTHLY REVERSAL FREQUENCY")
display(monthly)

# -----------------------------
# 12. REVERSAL MAGNITUDE
# -----------------------------

# Difference is Tmax - Tmin and is negative for a reversal.
# Magnitude is its absolute value, in degrees Celsius.

magnitude_summary = (
    events_df.groupby("model")
    .agg(
        reversal_count=("magnitude_C", "size"),
        mean_magnitude_C=("magnitude_C", "mean"),
        median_magnitude_C=("magnitude_C", "median"),
        std_magnitude_C=("magnitude_C", "std"),
        minimum_magnitude_C=("magnitude_C", "min"),
        maximum_magnitude_C=("magnitude_C", "max"),
        p90_magnitude_C=(
            "magnitude_C",
            lambda x: x.quantile(0.90)
        ),
        p95_magnitude_C=(
            "magnitude_C",
            lambda x: x.quantile(0.95)
        )
    )
    .reset_index()
)

magnitude_file = (
    OUTPUT_DIR / "Reversal_Magnitude_Summary.csv"
)

magnitude_summary.to_csv(
    magnitude_file,
    index=False
)

print("\nREVERSAL MAGNITUDE SUMMARY")
display(magnitude_summary)

# -----------------------------
# 13. MAGNITUDE BINS
# -----------------------------

bins = [
    0,
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    np.inf
]

labels = [
    "0 to 0.1°C",
    "0.1 to 0.5°C",
    "0.5 to 1°C",
    "1 to 2°C",
    "2 to 5°C",
    "Above 5°C"
]

events_df["magnitude_class"] = pd.cut(
    events_df["magnitude_C"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

magnitude_distribution = (
    events_df.groupby(
        ["model", "magnitude_class"],
        observed=False
    )
    .size()
    .reset_index(name="event_count")
)

magnitude_distribution["percentage"] = (
    magnitude_distribution["event_count"] /
    magnitude_distribution.groupby(
        "model"
    )["event_count"].transform("sum") * 100
)

magnitude_distribution_file = (
    OUTPUT_DIR / "Reversal_Magnitude_Distribution.csv"
)

magnitude_distribution.to_csv(
    magnitude_distribution_file,
    index=False
)

print("\nMAGNITUDE DISTRIBUTION")
display(magnitude_distribution)

# -----------------------------
# 14. SEASONAL FREQUENCY CHART
# -----------------------------

fig, ax = plt.subplots(figsize=(10, 6))

for model in MODELS:

    subset = seasonal[
        seasonal["model"] == model
    ].sort_values("season")

    ax.plot(
        subset["season"].astype(str),
        subset["frequency_percent"],
        marker="o",
        linewidth=2,
        label=model
    )

ax.set_title(
    "Seasonal Frequency of Tmax < Tmin Reversals"
)

ax.set_xlabel("Season")
ax.set_ylabel("Reversal frequency (%)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

seasonal_png = (
    OUTPUT_DIR / "Seasonal_Reversal_Frequency.png"
)

plt.savefig(
    seasonal_png,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# 15. ANNUAL FREQUENCY CHART
# -----------------------------

fig, ax = plt.subplots(figsize=(13, 6))

for model in MODELS:

    subset = annual[
        annual["model"] == model
    ].sort_values("year")

    ax.plot(
        subset["year"],
        subset["frequency_percent"],
        linewidth=1.3,
        label=model
    )

ax.set_title(
    "Annual Frequency of Tmax < Tmin Reversals"
)

ax.set_xlabel("Year")
ax.set_ylabel("Reversal frequency (%)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

annual_png = (
    OUTPUT_DIR / "Annual_Reversal_Frequency.png"
)

plt.savefig(
    annual_png,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# 16. MONTHLY FREQUENCY CHART
# -----------------------------

fig, ax = plt.subplots(figsize=(11, 6))

for model in MODELS:

    subset = monthly[
        monthly["model"] == model
    ].sort_values("month")

    ax.plot(
        subset["month"],
        subset["frequency_percent"],
        marker="o",
        linewidth=1.8,
        label=model
    )

ax.set_xticks(range(1, 13))
ax.set_xlabel("Calendar month")
ax.set_ylabel("Reversal frequency (%)")
ax.set_title(
    "Monthly Frequency of Tmax < Tmin Reversals"
)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

monthly_png = (
    OUTPUT_DIR / "Monthly_Reversal_Frequency.png"
)

plt.savefig(
    monthly_png,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# 17. MAGNITUDE BOXPLOT
# -----------------------------

fig, ax = plt.subplots(figsize=(9, 6))

box_data = [
    events_df.loc[
        events_df["model"] == model,
        "magnitude_C"
    ].values
    for model in MODELS
]

ax.boxplot(
    box_data,
    tick_labels=MODELS,
    showfliers=False
)

ax.set_ylabel("|Tmax - Tmin| (°C)")
ax.set_title(
    "Magnitude of Non-sentinel Temperature Reversals"
)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

magnitude_png = (
    OUTPUT_DIR / "Reversal_Magnitude_Boxplot.png"
)

plt.savefig(
    magnitude_png,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# 18. SAVE EVENT INVENTORY
# -----------------------------

events_file = (
    OUTPUT_DIR / "All_NonSentinel_Reversal_Events.csv"
)

events_df = events_df.sort_values(
    ["model", "Date", "grid_index"]
)

events_df.to_csv(
    events_file,
    index=False
)

# -----------------------------
# 19. HIGHEST FREQUENCY SEASONS
# -----------------------------

highest_season = (
    seasonal.sort_values(
        "frequency_percent",
        ascending=False
    )
    .groupby("model")
    .head(1)
)

print("\nHIGHEST REVERSAL-FREQUENCY SEASON BY MODEL")
display(highest_season)

# -----------------------------
# 20. HIGHEST FREQUENCY YEARS
# -----------------------------

highest_year = (
    annual.sort_values(
        "frequency_percent",
        ascending=False
    )
    .groupby("model")
    .head(5)
    .sort_values(
        ["model", "frequency_percent"],
        ascending=[True, False]
    )
)

print("\nFIVE HIGHEST-FREQUENCY YEARS PER MODEL")
display(highest_year)

# -----------------------------
# 21. FINAL OUTPUT INVENTORY
# -----------------------------

print("\n" + "=" * 65)
print("TEMPORAL AND MAGNITUDE QC COMPLETED")
print("=" * 65)

print("Output folder:", OUTPUT_DIR)

for f in [
    seasonal_file,
    annual_file,
    monthly_file,
    magnitude_file,
    magnitude_distribution_file,
    events_file,
    seasonal_png,
    annual_png,
    monthly_png,
    magnitude_png
]:
    print(f)

print("\nTotal events:", len(events_df))
print("Models:", events_df["model"].nunique())
print("Grid cells:", events_df["grid_index"].nunique())

print(
    "\nReminder: these are source-level QC flags. "
    "No values were corrected or removed."
)


**[orig #229]** Reversal event clustering QC (CanESM5, INM-CM4-8, INM-CM5-0)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ================================================================
# ALWAR CMIP6: TEMPERATURE REVERSAL EVENT CLUSTERING QC
# Models: CanESM5, INM-CM4-8, INM-CM5-0
# Scenario: SSP245
#
# Analyses:
# 1. Same-date clustering across grid cells
# 2. Consecutive-day persistence within each grid cell
# 3. Spatial-neighbour co-occurrence on the same date
# 4. Monthly and annual event clustering summaries
#
# No climate values are corrected or deleted.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

INPUT_DIR = os.path.join(
    BASE, "Climate_Indicators",
    "Reversal_Temporal_Magnitude_QC"
)

EVENT_FILE = os.path.join(
    INPUT_DIR, "All_NonSentinel_Reversal_Events.csv"
)

WEIGHT_FILE = os.path.join(
    BASE, "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = os.path.join(
    BASE, "Climate_Indicators",
    "Reversal_Event_Clustering_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(EVENT_FILE):
    raise FileNotFoundError(f"Event inventory not found: {EVENT_FILE}")

if not os.path.exists(WEIGHT_FILE):
    raise FileNotFoundError(f"Grid weight file not found: {WEIGHT_FILE}")

# ------------------------------------------------
# 2. LOAD AND STANDARDISE EVENT DATA
# ------------------------------------------------

events = pd.read_csv(EVENT_FILE)

print("Event inventory columns:")
print(events.columns.tolist())

# Standardise column names
events.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in events.columns
]

def find_column(possible_names, dataframe):
    for name in possible_names:
        if name in dataframe.columns:
            return name
    return None

date_col = find_column(
    ["date", "datetime", "time", "event_date"], events
)

model_col = find_column(
    ["model", "model_name", "gcm"], events
)

grid_col = find_column(
    ["grid_index", "column_index_zero_based",
     "grid_cell", "cell_index", "column_index"], events
)

if date_col is None or model_col is None or grid_col is None:
    raise ValueError(
        "Could not identify required columns. "
        f"Available columns: {events.columns.tolist()}"
    )

events = events.rename(columns={
    date_col: "Date",
    model_col: "model",
    grid_col: "grid_index"
})

events["Date"] = pd.to_datetime(events["Date"], errors="coerce")
events["grid_index"] = pd.to_numeric(
    events["grid_index"], errors="coerce"
)

events = events.dropna(
    subset=["Date", "model", "grid_index"]
).copy()

events["grid_index"] = events["grid_index"].astype(int)
events["year"] = events["Date"].dt.year
events["month"] = events["Date"].dt.month

events = events[
    events["model"].isin(
        ["CanESM5", "INM-CM4-8", "INM-CM5-0"]
    )
].copy()

# Each model/grid-cell/date should have at most one event.
duplicate_count = events.duplicated(
    subset=["model", "grid_index", "Date"]
).sum()

print("\nDuplicate model-grid-date records:", duplicate_count)

if duplicate_count > 0:
    print("Removing duplicate event records for clustering only.")
    events = events.drop_duplicates(
        subset=["model", "grid_index", "Date"]
    )

events = events.sort_values(
    ["model", "grid_index", "Date"]
).reset_index(drop=True)

print("\nLoaded event records:", len(events))
print("Models:", sorted(events["model"].unique()))
print("Grid cells represented:", events["grid_index"].nunique())
print("Date range:", events["Date"].min(), "to", events["Date"].max())

# ------------------------------------------------
# 3. LOAD GRID COORDINATES
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

weights.columns = [
    str(c).strip().lower().replace(" ", "_")
    for c in weights.columns
]

required_weight_cols = [
    "column_index_zero_based", "longitude", "latitude"
]

for col in required_weight_cols:
    if col not in weights.columns:
        raise ValueError(
            f"Missing {col} in weight file. "
            f"Available columns: {weights.columns.tolist()}"
        )

grid_coords = (
    weights[
        ["column_index_zero_based", "longitude", "latitude"]
    ]
    .drop_duplicates()
    .rename(columns={
        "column_index_zero_based": "grid_index"
    })
)

grid_coords["grid_index"] = grid_coords["grid_index"].astype(int)

events = events.merge(
    grid_coords,
    on="grid_index",
    how="left",
    validate="many_to_one"
)

if events["longitude"].isna().any():
    missing_grids = sorted(
        events.loc[
            events["longitude"].isna(), "grid_index"
        ].unique()
    )
    raise ValueError(
        f"Grid coordinates missing for indices: {missing_grids}"
    )

print("\nGrid coordinate matching successful.")

# ------------------------------------------------
# 4. SAME-DATE CLUSTERING ACROSS GRID CELLS
# ------------------------------------------------

date_cluster = (
    events.groupby(["model", "Date"])
    .agg(
        reversal_grid_cells=("grid_index", "nunique"),
        total_reversal_events=("grid_index", "size")
    )
    .reset_index()
)

date_cluster["year"] = date_cluster["Date"].dt.year
date_cluster["month"] = date_cluster["Date"].dt.month

# Count how many grid cells had a reversal on each model-date.
date_cluster["cluster_class"] = pd.cut(
    date_cluster["reversal_grid_cells"],
    bins=[0, 1, 2, 4, 8, 22],
    labels=[
        "1 grid cell",
        "2 grid cells",
        "3–4 grid cells",
        "5–8 grid cells",
        "9–22 grid cells"
    ],
    include_lowest=True
)

date_cluster.to_csv(
    os.path.join(OUTPUT_DIR, "Same_Date_Grid_Cell_Clustering.csv"),
    index=False
)

same_date_summary = (
    date_cluster.groupby(
        ["model", "cluster_class"],
        observed=False
    )
    .agg(
        number_of_dates=("Date", "nunique"),
        total_events=("total_reversal_events", "sum")
    )
    .reset_index()
)

same_date_summary.to_csv(
    os.path.join(OUTPUT_DIR, "Same_Date_Clustering_Summary.csv"),
    index=False
)

# Number of dates with simultaneous events in multiple cells.
simultaneous_summary = (
    date_cluster.groupby("model")
    .agg(
        dates_with_events=("Date", "nunique"),
        dates_with_2_or_more_cells=(
            "reversal_grid_cells",
            lambda x: int((x >= 2).sum())
        ),
        dates_with_3_or_more_cells=(
            "reversal_grid_cells",
            lambda x: int((x >= 3).sum())
        ),
        dates_with_5_or_more_cells=(
            "reversal_grid_cells",
            lambda x: int((x >= 5).sum())
        ),
        maximum_cells_on_one_date=(
            "reversal_grid_cells", "max"
        ),
        mean_cells_per_event_date=(
            "reversal_grid_cells", "mean"
        )
    )
    .reset_index()
)

simultaneous_summary.to_csv(
    os.path.join(OUTPUT_DIR, "Simultaneous_Date_Summary.csv"),
    index=False
)

# ------------------------------------------------
# 5. CONSECUTIVE-DAY PERSISTENCE
# ------------------------------------------------

# For each model and grid cell, identify runs of consecutive
# calendar dates on which a reversal occurred.

persistence_records = []

for (model, grid), group in events.groupby(
    ["model", "grid_index"]
):
    dates = pd.Series(
        sorted(group["Date"].drop_duplicates())
    )

    if len(dates) == 0:
        continue

    day_gaps = dates.diff().dt.days
    run_ids = day_gaps.ne(1).cumsum()

    temp = pd.DataFrame({
        "Date": dates,
        "run_id": run_ids
    })

    for run_id, run in temp.groupby("run_id"):
        persistence_records.append({
            "model": model,
            "grid_index": grid,
            "run_start": run["Date"].min(),
            "run_end": run["Date"].max(),
            "consecutive_event_days": len(run)
        })

runs = pd.DataFrame(persistence_records)

runs.to_csv(
    os.path.join(OUTPUT_DIR, "Consecutive_Day_Event_Runs.csv"),
    index=False
)

if not runs.empty:
    persistence_summary = (
        runs.groupby("model")
        .agg(
            number_of_event_runs=("consecutive_event_days", "size"),
            single_day_runs=(
                "consecutive_event_days",
                lambda x: int((x == 1).sum())
            ),
            two_day_or_longer_runs=(
                "consecutive_event_days",
                lambda x: int((x >= 2).sum())
            ),
            three_day_or_longer_runs=(
                "consecutive_event_days",
                lambda x: int((x >= 3).sum())
            ),
            maximum_run_days=("consecutive_event_days", "max"),
            mean_run_days=("consecutive_event_days", "mean")
        )
        .reset_index()
    )
else:
    persistence_summary = pd.DataFrame()

persistence_summary.to_csv(
    os.path.join(OUTPUT_DIR, "Persistence_Summary.csv"),
    index=False
)

# ------------------------------------------------
# 6. SPATIAL NEIGHBOUR CO-OCCURRENCE
# ------------------------------------------------

# Define neighbouring grid cells using the source 0.25-degree
# grid. Cells touching by an edge or corner are neighbours.
# The tolerance allows for floating-point coordinate differences.

coord_lookup = (
    grid_coords.set_index("grid_index")[["longitude", "latitude"]]
    .to_dict("index")
)

grid_ids = sorted(coord_lookup.keys())
neighbours = {g: set() for g in grid_ids}

for g1 in grid_ids:
    lon1 = coord_lookup[g1]["longitude"]
    lat1 = coord_lookup[g1]["latitude"]

    for g2 in grid_ids:
        if g1 == g2:
            continue

        lon2 = coord_lookup[g2]["longitude"]
        lat2 = coord_lookup[g2]["latitude"]

        lon_gap = abs(lon1 - lon2)
        lat_gap = abs(lat1 - lat2)

        if (
            lon_gap <= 0.251
            and lat_gap <= 0.251
            and lon_gap + lat_gap > 0
        ):
            neighbours[g1].add(g2)

spatial_records = []

for (model, date), group in events.groupby(["model", "Date"]):
    active_grids = set(group["grid_index"].astype(int))

    cells_with_active_neighbour = set()

    for grid in active_grids:
        if neighbours.get(grid, set()).intersection(active_grids):
            cells_with_active_neighbour.add(grid)

    spatial_records.append({
        "model": model,
        "Date": date,
        "active_grid_cells": len(active_grids),
        "cells_with_same_date_neighbour": len(
            cells_with_active_neighbour
        ),
        "has_neighbour_cooccurrence": (
            len(cells_with_active_neighbour) > 0
        )
    })

spatial_daily = pd.DataFrame(spatial_records)

spatial_daily.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Same_Date_Spatial_Neighbour_Cooccurrence.csv"
    ),
    index=False
)

spatial_summary = (
    spatial_daily.groupby("model")
    .agg(
        event_dates=("Date", "nunique"),
        dates_with_neighbour_cooccurrence=(
            "has_neighbour_cooccurrence", "sum"
        ),
        maximum_active_cells=("active_grid_cells", "max"),
        maximum_neighbour_cells=(
            "cells_with_same_date_neighbour", "max"
        )
    )
    .reset_index()
)

spatial_summary["neighbour_cooccurrence_percent"] = (
    100 * spatial_summary["dates_with_neighbour_cooccurrence"]
    / spatial_summary["event_dates"]
)

spatial_summary.to_csv(
    os.path.join(OUTPUT_DIR, "Spatial_Neighbour_Summary.csv"),
    index=False
)

# ------------------------------------------------
# 7. MONTHLY AND ANNUAL CLUSTERING SUMMARIES
# ------------------------------------------------

monthly_cluster = (
    date_cluster.groupby(["model", "month"])
    .agg(
        event_dates=("Date", "nunique"),
        total_reversal_events=("total_reversal_events", "sum"),
        mean_active_cells=("reversal_grid_cells", "mean"),
        maximum_active_cells=("reversal_grid_cells", "max")
    )
    .reset_index()
)

monthly_cluster.to_csv(
    os.path.join(OUTPUT_DIR, "Monthly_Clustering_Summary.csv"),
    index=False
)

annual_cluster = (
    date_cluster.groupby(["model", "year"])
    .agg(
        event_dates=("Date", "nunique"),
        total_reversal_events=("total_reversal_events", "sum"),
        mean_active_cells=("reversal_grid_cells", "mean"),
        maximum_active_cells=("reversal_grid_cells", "max")
    )
    .reset_index()
)

annual_cluster.to_csv(
    os.path.join(OUTPUT_DIR, "Annual_Clustering_Summary.csv"),
    index=False
)

# ------------------------------------------------
# 8. PLOTS
# ------------------------------------------------

# A. Same-date grid-cell clustering distribution
pivot = (
    date_cluster.groupby(
        ["model", "reversal_grid_cells"]
    ).size().unstack(fill_value=0)
)

ax = pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6)
)

ax.set_title("Same-date temperature reversal clustering")
ax.set_xlabel("Climate model")
ax.set_ylabel("Number of event dates")
ax.legend(
    title="Number of affected grid cells",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "Same_Date_Clustering.png"),
    dpi=300
)
plt.show()
plt.close()

# B. Consecutive-day run distribution
if not runs.empty:
    run_plot = (
        runs.groupby(["model", "consecutive_event_days"])
        .size()
        .unstack(fill_value=0)
    )

    ax = run_plot.plot(
        kind="bar",
        figsize=(11, 6)
    )
    ax.set_title("Consecutive-day reversal event runs")
    ax.set_xlabel("Climate model")
    ax.set_ylabel("Number of grid-cell event runs")
    ax.legend(
        title="Consecutive days",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "Consecutive_Day_Runs.png"),
        dpi=300
    )
    plt.show()
    plt.close()

# C. Annual event-date counts
for model in sorted(annual_cluster["model"].unique()):
    subset = annual_cluster[
        annual_cluster["model"] == model
    ]

    plt.figure(figsize=(12, 4))
    plt.bar(subset["year"], subset["event_dates"])
    plt.title(f"Annual reversal event dates: {model}")
    plt.xlabel("Year")
    plt.ylabel("Number of dates with reversals")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"Annual_Event_Dates_{model}.png"
        ),
        dpi=300
    )
    plt.show()
    plt.close()

# ------------------------------------------------
# 9. DISPLAY RESULTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("SAME-DATE CLUSTERING SUMMARY")
print("=" * 70)
display(simultaneous_summary)

print("\n" + "=" * 70)
print("CONSECUTIVE-DAY PERSISTENCE SUMMARY")
print("=" * 70)
display(persistence_summary)

print("\n" + "=" * 70)
print("SPATIAL NEIGHBOUR CO-OCCURRENCE SUMMARY")
print("=" * 70)
display(spatial_summary)

print("\n" + "=" * 70)
print("MONTHLY CLUSTERING SUMMARY")
print("=" * 70)
display(monthly_cluster)

print("\n" + "=" * 70)
print("QC CLUSTERING COMPLETED")
print("=" * 70)
print("Total event records:", len(events))
print("Total event dates by model:")
print(date_cluster.groupby("model")["Date"].nunique())
print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(filename)


**[orig #230]** Identical temperature-pair test (needs `events` and `OUTPUT_DIR` from the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ------------------------------------------------
# 6. IDENTICAL TEMPERATURE PAIR TEST (PANDAS-COMPATIBLE)
# ------------------------------------------------

# Count unique Tmax/Tmin combinations for each model-date.
# No temperature observations are changed.

pair_signature = (
    events.groupby(["model", "Date"])
    .apply(
        lambda g: pd.Series({
            "affected_cells": g["grid_index"].nunique(),
            "unique_temperature_pairs": (
                g[["Tmax_C", "Tmin_C"]]
                .drop_duplicates()
                .shape[0]
            )
        }),
        include_groups=False
    )
    .reset_index()
)

pair_signature["repeated_temperature_pairs"] = (
    pair_signature["affected_cells"]
    - pair_signature["unique_temperature_pairs"]
)

pair_signature["unique_pair_percentage"] = (
    100 * pair_signature["unique_temperature_pairs"]
    / pair_signature["affected_cells"]
)

pair_signature.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Temperature_Pair_Uniqueness.csv"
    ),
    index=False
)

print("Temperature-pair uniqueness analysis completed.")
display(pair_signature.head(15))

print("\nDates with repeated temperature pairs:")
display(
    pair_signature[
        pair_signature["repeated_temperature_pairs"] > 0
    ].head(20)
)


**[orig #232]** Find and load the reversal event inventory; model-level summary  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "Alwar_grid_analysis/Climate_Indicators"
)

# Search recursively
matches = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file == "Temperature_Pair_Uniqueness.csv":
            matches.append(os.path.join(root, file))

if not matches:
    raise FileNotFoundError(
        "Temperature_Pair_Uniqueness.csv was not found. "
        "Please run the corrected Step 6 again and check its save path."
    )

INPUT_FILE = matches[0]

print("Input file found:")
print(INPUT_FILE)

# Load data
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])

# Model-level summary
summary = (
    df.groupby("model")
    .agg(
        total_event_dates=("Date", "nunique"),
        total_affected_cell_records=("affected_cells", "sum"),
        total_unique_pair_records=("unique_temperature_pairs", "sum"),
        total_repeated_pair_records=("repeated_temperature_pairs", "sum"),
        dates_with_repeated_pairs=(
            "repeated_temperature_pairs",
            lambda x: (x > 0).sum()
        ),
        maximum_affected_cells=("affected_cells", "max"),
        mean_unique_pair_percentage=("unique_pair_percentage", "mean"),
        minimum_unique_pair_percentage=("unique_pair_percentage", "min")
    )
    .reset_index()
)

summary["overall_unique_pair_percentage"] = (
    100 * summary["total_unique_pair_records"]
    / summary["total_affected_cell_records"]
)

summary["percentage_dates_with_repeated_pairs"] = (
    100 * summary["dates_with_repeated_pairs"]
    / summary["total_event_dates"]
)

# Save beside the input file
OUTPUT_FILE = os.path.join(
    os.path.dirname(INPUT_FILE),
    "Temperature_Pair_Uniqueness_Model_Summary.csv"
)

summary.to_csv(OUTPUT_FILE, index=False)

print("\nAnalysis completed successfully.")
print("Output saved:", OUTPUT_FILE)

display(summary.round(3))

**[orig #233]** Set paths to the event inventory  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "Alwar_grid_analysis/Climate_Indicators"
)

INPUT_FILE = os.path.join(
    BASE_DIR,
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

print("Original event inventory exists:", os.path.exists(INPUT_FILE))

if os.path.exists(INPUT_FILE):
    print("File size (MB):", round(os.path.getsize(INPUT_FILE)/1024**2, 2))
    print("Path:", INPUT_FILE)

**[orig #234]** Temperature-pair uniqueness (needs the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Cross_Model_Synchronisation_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load original event inventory
events = pd.read_csv(INPUT_FILE)

events["Date"] = pd.to_datetime(events["Date"])

print("Total events:", len(events))
print("Models:", events["model"].unique())

# Check required columns
required = [
    "model", "Date", "grid_index", "Tmax_C", "Tmin_C"
]

missing = [c for c in required if c not in events.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Temperature-pair uniqueness
pair_signature = (
    events.groupby(["model", "Date"])
    .apply(
        lambda g: pd.Series({
            "affected_cells": g["grid_index"].nunique(),
            "unique_temperature_pairs": (
                g[["Tmax_C", "Tmin_C"]]
                .drop_duplicates()
                .shape[0]
            )
        })
    )
    .reset_index()
)

pair_signature["repeated_temperature_pairs"] = (
    pair_signature["affected_cells"]
    - pair_signature["unique_temperature_pairs"]
)

pair_signature["unique_pair_percentage"] = (
    100 * pair_signature["unique_temperature_pairs"]
    / pair_signature["affected_cells"]
)

# Save detailed results
pair_file = os.path.join(
    OUTPUT_DIR,
    "Temperature_Pair_Uniqueness.csv"
)

pair_signature.to_csv(pair_file, index=False)

# Model-level summary
summary = (
    pair_signature.groupby("model")
    .agg(
        total_event_dates=("Date", "nunique"),
        total_affected_cell_records=("affected_cells", "sum"),
        total_unique_pair_records=("unique_temperature_pairs", "sum"),
        total_repeated_pair_records=("repeated_temperature_pairs", "sum"),
        dates_with_repeated_pairs=(
            "repeated_temperature_pairs",
            lambda x: (x > 0).sum()
        ),
        mean_unique_pair_percentage=("unique_pair_percentage", "mean"),
        minimum_unique_pair_percentage=("unique_pair_percentage", "min")
    )
    .reset_index()
)

summary["overall_unique_pair_percentage"] = (
    100 * summary["total_unique_pair_records"]
    / summary["total_affected_cell_records"]
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Temperature_Pair_Uniqueness_Model_Summary.csv"
)

summary.to_csv(summary_file, index=False)

print("\nSUCCESS: Both files saved.")
print(pair_file)
print(summary_file)

display(summary.round(3))

**[orig #238]** Load the non-sentinel reversal events  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

CLIMATE_QC_DIR = os.path.join(BASE_DIR, "Climate_Indicators")

INPUT_FILE = os.path.join(
    CLIMATE_QC_DIR,
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

OUTPUT_DIR = os.path.join(
    CLIMATE_QC_DIR,
    "Cross_Model_Synchronisation_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("BASE DIRECTORY:", os.path.exists(BASE_DIR))
print("EVENT INVENTORY:", os.path.exists(INPUT_FILE))
print("OUTPUT DIRECTORY:", OUTPUT_DIR)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

events = pd.read_csv(INPUT_FILE)

events["Date"] = pd.to_datetime(events["Date"])

print("\nTotal reversal events:", len(events))
print("\nEvents by model:")
display(events.groupby("model").size().reset_index(name="events"))

print("\nDate range:")
print(events["Date"].min(), "to", events["Date"].max())

print("\nColumns:")
print(events.columns.tolist())

**[orig #239]** Temperature-pair uniqueness on the loaded events (needs the cell above)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import os

required = [
    "model", "Date", "grid_index", "Tmax_C", "Tmin_C"
]

missing = [c for c in required if c not in events.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

pair_signature = (
    events.groupby(["model", "Date"])
    .apply(
        lambda g: pd.Series({
            "affected_cells": g["grid_index"].nunique(),
            "unique_temperature_pairs": (
                g[["Tmax_C", "Tmin_C"]]
                .drop_duplicates()
                .shape[0]
            )
        })
    )
    .reset_index()
)

pair_signature["repeated_temperature_pairs"] = (
    pair_signature["affected_cells"]
    - pair_signature["unique_temperature_pairs"]
)

pair_signature["unique_pair_percentage"] = (
    100 * pair_signature["unique_temperature_pairs"]
    / pair_signature["affected_cells"]
)

pair_file = os.path.join(
    OUTPUT_DIR,
    "Temperature_Pair_Uniqueness.csv"
)

pair_signature.to_csv(pair_file, index=False)

summary = (
    pair_signature.groupby("model")
    .agg(
        total_event_dates=("Date", "nunique"),
        total_affected_cell_records=("affected_cells", "sum"),
        total_unique_pair_records=("unique_temperature_pairs", "sum"),
        total_repeated_pair_records=("repeated_temperature_pairs", "sum"),
        dates_with_repeated_pairs=(
            "repeated_temperature_pairs",
            lambda x: (x > 0).sum()
        ),
        mean_unique_pair_percentage=("unique_pair_percentage", "mean"),
        minimum_unique_pair_percentage=("unique_pair_percentage", "min")
    )
    .reset_index()
)

summary["overall_unique_pair_percentage"] = (
    100 * summary["total_unique_pair_records"]
    / summary["total_affected_cell_records"]
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Temperature_Pair_Uniqueness_Model_Summary.csv"
)

summary.to_csv(summary_file, index=False)

print("Temperature-pair analysis successfully saved.")
print("\nDetailed file:", pair_file)
print("Summary file:", summary_file)

display(summary.round(3))

## 4. Cross-model synchronisation

**[orig #240]** Cross-model synchronisation of reversal dates  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# CROSS-MODEL TEMPERATURE REVERSAL SYNCHRONISATION
# ALWAR DISTRICT | SSP245 | 2015–2100
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

QC_DIR = os.path.join(BASE_DIR, "Climate_Indicators")

INPUT_FILE = os.path.join(
    QC_DIR,
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

OUTPUT_DIR = os.path.join(
    QC_DIR,
    "Cross_Model_Synchronisation_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

# ------------------------------------------------------------
# 2. LOAD EVENT INVENTORY
# ------------------------------------------------------------

events = pd.read_csv(INPUT_FILE)

required = [
    "model", "Date", "grid_index", "Tmax_C", "Tmin_C"
]

missing = [c for c in required if c not in events.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

events["Date"] = pd.to_datetime(events["Date"]).dt.normalize()

models = ["CanESM5", "INM-CM4-8", "INM-CM5-0"]

events = events[events["model"].isin(models)].copy()

# One record per model/date/grid cell for date-level analysis.
events = events.drop_duplicates(
    subset=["model", "Date", "grid_index"]
).copy()

# Positive gap means Tmin exceeds Tmax.
events["reversal_gap_C"] = events["Tmin_C"] - events["Tmax_C"]

print("Models:", events["model"].unique())
print("Unique model-date-grid events:", len(events))
print("Date range:", events["Date"].min(), "to", events["Date"].max())

# ------------------------------------------------------------
# 3. CREATE MODEL-DATE INVENTORY
# ------------------------------------------------------------

model_date = (
    events.groupby(["Date", "model"])
    .agg(
        affected_cells=("grid_index", "nunique"),
        mean_Tmax_C=("Tmax_C", "mean"),
        mean_Tmin_C=("Tmin_C", "mean"),
        mean_reversal_gap_C=("reversal_gap_C", "mean"),
        median_reversal_gap_C=("reversal_gap_C", "median"),
        maximum_reversal_gap_C=("reversal_gap_C", "max"),
        minimum_reversal_gap_C=("reversal_gap_C", "min")
    )
    .reset_index()
)

model_date_file = os.path.join(
    OUTPUT_DIR,
    "Cross_Model_Date_Inventory.csv"
)

model_date.to_csv(model_date_file, index=False)

# ------------------------------------------------------------
# 4. CREATE DAILY MODEL PRESENCE TABLE
# ------------------------------------------------------------

presence = (
    model_date.pivot(
        index="Date",
        columns="model",
        values="affected_cells"
    )
    .reindex(columns=models)
)

presence = presence.fillna(0).astype(int)

for model in models:
    presence[model + "_event"] = (
        presence[model] > 0
    ).astype(int)

event_columns = [m + "_event" for m in models]

presence["models_with_events"] = presence[event_columns].sum(axis=1)

presence["synchronisation_category"] = (
    presence["models_with_events"]
    .map({
        1: "One model",
        2: "Two models",
        3: "All three models"
    })
)

daily_file = os.path.join(
    OUTPUT_DIR,
    "Daily_Cross_Model_Synchronisation.csv"
)

presence.reset_index().to_csv(daily_file, index=False)

# ------------------------------------------------------------
# 5. CROSS-MODEL DATE OVERLAP
# ------------------------------------------------------------

date_sets = {
    model: set(model_date.loc[
        model_date["model"] == model, "Date"
    ])
    for model in models
}

overlap_rows = []

for model_a in models:
    for model_b in models:
        common = date_sets[model_a] & date_sets[model_b]

        overlap_rows.append({
            "model_1": model_a,
            "model_2": model_b,
            "dates_model_1": len(date_sets[model_a]),
            "dates_model_2": len(date_sets[model_b]),
            "shared_dates": len(common),
            "percentage_of_model_1_dates": (
                100 * len(common) / len(date_sets[model_a])
                if date_sets[model_a] else np.nan
            ),
            "percentage_of_model_2_dates": (
                100 * len(common) / len(date_sets[model_b])
                if date_sets[model_b] else np.nan
            )
        })

overlap = pd.DataFrame(overlap_rows)

overlap_file = os.path.join(
    OUTPUT_DIR,
    "Cross_Model_Date_Overlap.csv"
)

overlap.to_csv(overlap_file, index=False)

# ------------------------------------------------------------
# 6. SYNCHRONISATION SUMMARY
# ------------------------------------------------------------

union_dates = set.union(*date_sets.values())
common_all = set.intersection(*date_sets.values())

summary_rows = []

for model in models:
    summary_rows.append({
        "model": model,
        "event_dates": len(date_sets[model]),
        "percentage_of_union_dates": (
            100 * len(date_sets[model]) / len(union_dates)
            if union_dates else np.nan
        )
    })

summary = pd.DataFrame(summary_rows)

category_counts = (
    presence["models_with_events"]
    .value_counts()
    .reindex([1, 2, 3], fill_value=0)
)

sync_summary = pd.DataFrame([{
    "total_unique_event_dates_across_models": len(union_dates),
    "dates_with_one_model": int(category_counts.loc[1]),
    "dates_with_two_models": int(category_counts.loc[2]),
    "dates_with_all_three_models": int(category_counts.loc[3]),
    "dates_shared_by_all_three_models": len(common_all),
    "percentage_union_dates_shared_by_all_three": (
        100 * len(common_all) / len(union_dates)
        if union_dates else np.nan
    )
}])

summary_file = os.path.join(
    OUTPUT_DIR,
    "Cross_Model_Synchrony_Summary.csv"
)

sync_summary.to_csv(summary_file, index=False)

# ------------------------------------------------------------
# 7. TEMPERATURE SIGNATURE ON SHARED DATES
# ------------------------------------------------------------

signature = model_date[
    model_date["Date"].isin(common_all)
].copy()

signature_file = os.path.join(
    OUTPUT_DIR,
    "Event_Date_Temperature_Signatures.csv"
)

signature.to_csv(signature_file, index=False)

# ------------------------------------------------------------
# 8. MONTHLY CROSS-MODEL EVENT SIGNATURES
# ------------------------------------------------------------

model_date["Year"] = model_date["Date"].dt.year
model_date["Month"] = model_date["Date"].dt.month

monthly = (
    model_date.groupby(["model", "Month"])
    .agg(
        event_dates=("Date", "nunique"),
        affected_cell_records=("affected_cells", "sum"),
        mean_reversal_gap_C=("mean_reversal_gap_C", "mean")
    )
    .reset_index()
)

monthly.to_csv(
    os.path.join(OUTPUT_DIR, "Monthly_Event_Signatures.csv"),
    index=False
)

# ------------------------------------------------------------
# 9. ANNUAL CROSS-MODEL EVENT SIGNATURES
# ------------------------------------------------------------

annual = (
    model_date.groupby(["model", "Year"])
    .agg(
        event_dates=("Date", "nunique"),
        affected_cell_records=("affected_cells", "sum"),
        mean_reversal_gap_C=("mean_reversal_gap_C", "mean")
    )
    .reset_index()
)

annual.to_csv(
    os.path.join(OUTPUT_DIR, "Annual_Event_Signatures.csv"),
    index=False
)

# ------------------------------------------------------------
# 10. PLOT MODEL DATE OVERLAP
# ------------------------------------------------------------

overlap_matrix = pd.DataFrame(
    index=models,
    columns=models,
    dtype=float
)

for a in models:
    for b in models:
        overlap_matrix.loc[a, b] = len(date_sets[a] & date_sets[b])

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(overlap_matrix.values, aspect="auto")

ax.set_xticks(range(len(models)))
ax.set_yticks(range(len(models)))
ax.set_xticklabels(models, rotation=30, ha="right")
ax.set_yticklabels(models)

for i in range(len(models)):
    for j in range(len(models)):
        ax.text(
            j, i,
            int(overlap_matrix.iloc[i, j]),
            ha="center",
            va="center"
        )

ax.set_title("Shared Temperature Reversal Dates Across Models")
ax.set_xlabel("Model")
ax.set_ylabel("Model")

fig.colorbar(im, ax=ax, label="Number of shared dates")
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "Cross_Model_Date_Overlap.png"),
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# 11. PLOT DAILY SYNCHRONISATION
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))

category_counts.plot(
    kind="bar",
    ax=ax
)

ax.set_xticklabels(
    ["One model", "Two models", "All three models"],
    rotation=0
)

ax.set_ylabel("Number of calendar dates")
ax.set_title("Cross-Model Temperature Reversal Synchronisation")
ax.set_xlabel("Number of models with reversals")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "Cross_Model_Synchrony.png"),
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# 12. FINAL OUTPUT
# ------------------------------------------------------------

print("\n============================================")
print("CROSS-MODEL SYNCHRONISATION ANALYSIS COMPLETE")
print("============================================")

print("\nModel event-date summary:")
display(summary.round(3))

print("\nCross-model synchronisation summary:")
display(sync_summary.round(3))

print("\nPairwise date overlap:")
display(overlap.round(3))

print("\nDates shared by all three models:", len(common_all))

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print(file)

**[orig #241]** Shared reversal dates: detailed event records  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

QC_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators"
)

INPUT_FILE = os.path.join(
    QC_DIR,
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

OUTPUT_DIR = os.path.join(
    QC_DIR,
    "Cross_Model_Synchronisation_QC"
)

events = pd.read_csv(INPUT_FILE)
events["Date"] = pd.to_datetime(events["Date"]).dt.normalize()

models = ["CanESM5", "INM-CM4-8", "INM-CM5-0"]

# Identify shared dates
date_sets = {
    model: set(
        events.loc[events["model"] == model, "Date"]
    )
    for model in models
}

shared_dates = (
    date_sets["CanESM5"]
    & date_sets["INM-CM4-8"]
)

print("Shared dates:", sorted(shared_dates))

# Extract detailed event records
shared_events = events[
    (events["Date"].isin(shared_dates))
    & (events["model"].isin(models))
].copy()

shared_events["reversal_gap_C"] = (
    shared_events["Tmin_C"] - shared_events["Tmax_C"]
)

# Model/date summary
summary = (
    shared_events.groupby(["Date", "model"])
    .agg(
        affected_cells=("grid_index", "nunique"),
        mean_Tmax_C=("Tmax_C", "mean"),
        mean_Tmin_C=("Tmin_C", "mean"),
        mean_reversal_gap_C=("reversal_gap_C", "mean"),
        maximum_reversal_gap_C=("reversal_gap_C", "max")
    )
    .reset_index()
)

# Save detailed outputs
shared_events.to_csv(
    os.path.join(OUTPUT_DIR, "Two_Shared_Dates_Event_Details.csv"),
    index=False
)

summary.to_csv(
    os.path.join(OUTPUT_DIR, "Two_Shared_Dates_Model_Comparison.csv"),
    index=False
)

print("\nShared-date model comparison:")
display(summary.round(4))

print("\nIndividual grid-cell events:")
display(
    shared_events[
        ["Date", "model", "grid_index",
         "Tmax_C", "Tmin_C", "reversal_gap_C"]
    ].sort_values(["Date", "model", "grid_index"])
)

**[orig #242]** Check the rebuilt grid files for the shared dates  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

GRID_DIR = os.path.join(
    BASE_DIR,
    "Alwar_daily_extracted_REBUILT"
)

WEIGHT_FILE = os.path.join(
    BASE_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

for model in ["CanESM5", "INM-CM4-8"]:

    filename = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMaxData_Alwar_REBUILT.csv"
    )

    print("\nMODEL:", model)
    print("File exists:", os.path.exists(filename))

    if os.path.exists(filename):
        df = pd.read_csv(filename, nrows=3)

        print("Number of columns:", len(df.columns))
        print("Column names:")
        print(df.columns.tolist())

        print("\nFirst three records:")
        display(df.iloc[:, :10])

print("\n\nSPATIAL WEIGHT FILE")

print("File exists:", os.path.exists(WEIGHT_FILE))

if os.path.exists(WEIGHT_FILE):
    weights = pd.read_csv(WEIGHT_FILE)

    print("Weight file columns:")
    print(weights.columns.tolist())

    print("\nFirst records:")
    display(weights.head())

## 5. Checks against the original source; sentinel values

**[orig #243]** Source-to-extraction verification for the two shared dates  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# SOURCE-TO-EXTRACTION VERIFICATION
# TWO SHARED TEMPERATURE REVERSAL DATES
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

EVENT_FILE = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

GRID_DIR = os.path.join(
    BASE_DIR,
    "Alwar_daily_extracted_REBUILT"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Cross_Model_Synchronisation_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD EVENTS
# ------------------------------------------------------------

events = pd.read_csv(EVENT_FILE)

events["Date"] = pd.to_datetime(
    events["Date"]
).dt.normalize()

shared_dates = pd.to_datetime([
    "2043-11-26",
    "2072-12-07"
])

events = events[
    events["Date"].isin(shared_dates)
].copy()

models = ["CanESM5", "INM-CM4-8"]

print("Events selected:", len(events))
print("Models:", events["model"].unique())

# ------------------------------------------------------------
# 3. VERIFY GRID COLUMN MAPPING
# ------------------------------------------------------------

results = []

for model in models:

    print("\nProcessing model:", model)

    tmax_file = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMaxData_Alwar_REBUILT.csv"
    )

    tmin_file = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMinData_Alwar_REBUILT.csv"
    )

    if not os.path.exists(tmax_file):
        raise FileNotFoundError(tmax_file)

    if not os.path.exists(tmin_file):
        raise FileNotFoundError(tmin_file)

    tmax = pd.read_csv(tmax_file)
    tmin = pd.read_csv(tmin_file)

    tmax["Date"] = pd.to_datetime(
        tmax["Date"]
    ).dt.normalize()

    tmin["Date"] = pd.to_datetime(
        tmin["Date"]
    ).dt.normalize()

    # Set date as index for direct lookup
    tmax = tmax.set_index("Date")
    tmin = tmin.set_index("Date")

    model_events = events[
        events["model"] == model
    ]

    for _, event in model_events.iterrows():

        date = event["Date"]

        grid_index = int(event["grid_index"])

        # Correct grid-column naming convention
        grid_column = f"grid_col_{grid_index}"

        if grid_column not in tmax.columns:
            raise ValueError(
                f"{grid_column} missing from Tmax file for {model}"
            )

        if grid_column not in tmin.columns:
            raise ValueError(
                f"{grid_column} missing from Tmin file for {model}"
            )

        if date not in tmax.index or date not in tmin.index:
            raise ValueError(
                f"Date missing: {model}, {date}"
            )

        extracted_tmax = float(
            tmax.loc[date, grid_column]
        )

        extracted_tmin = float(
            tmin.loc[date, grid_column]
        )

        event_tmax = float(event["Tmax_C"])
        event_tmin = float(event["Tmin_C"])

        results.append({

            "model": model,

            "Date": date,

            "grid_index": grid_index,

            "grid_column": grid_column,

            "event_Tmax_C": event_tmax,

            "extracted_Tmax_C": extracted_tmax,

            "event_Tmin_C": event_tmin,

            "extracted_Tmin_C": extracted_tmin,

            "Tmax_difference_C": (
                extracted_tmax - event_tmax
            ),

            "Tmin_difference_C": (
                extracted_tmin - event_tmin
            ),

            "event_reversal": (
                event_tmin > event_tmax
            ),

            "extracted_reversal": (
                extracted_tmin > extracted_tmax
            ),

            "extracted_reversal_gap_C": (
                extracted_tmin - extracted_tmax
            )
        })

# ------------------------------------------------------------
# 4. CREATE VERIFICATION TABLE
# ------------------------------------------------------------

verification = pd.DataFrame(results)

verification["Tmax_match"] = (
    verification["Tmax_difference_C"].abs() <= 1e-6
)

verification["Tmin_match"] = (
    verification["Tmin_difference_C"].abs() <= 1e-6
)

verification["both_values_match"] = (
    verification["Tmax_match"]
    & verification["Tmin_match"]
)

# ------------------------------------------------------------
# 5. SAVE DETAILED REPORT
# ------------------------------------------------------------

DETAIL_FILE = os.path.join(
    OUTPUT_DIR,
    "Shared_Date_Source_Extraction_Verification.csv"
)

verification.to_csv(
    DETAIL_FILE,
    index=False
)

# ------------------------------------------------------------
# 6. SUMMARY BY MODEL
# ------------------------------------------------------------

summary = (
    verification.groupby("model")
    .agg(
        total_records=("grid_index", "count"),

        Tmax_matches=("Tmax_match", "sum"),

        Tmin_matches=("Tmin_match", "sum"),

        complete_matches=("both_values_match", "sum"),

        maximum_Tmax_difference_C=(
            "Tmax_difference_C",
            lambda x: x.abs().max()
        ),

        maximum_Tmin_difference_C=(
            "Tmin_difference_C",
            lambda x: x.abs().max()
        ),

        extracted_reversal_records=(
            "extracted_reversal",
            "sum"
        )
    )
    .reset_index()
)

summary["matching_percentage"] = (
    100 * summary["complete_matches"]
    / summary["total_records"]
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "Shared_Date_Source_Extraction_Summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========================================")
print("SOURCE-TO-EXTRACTION VERIFICATION")
print("========================================")

print("\nDetailed verification:")
display(verification.round(6))

print("\nModel-level summary:")
display(summary.round(6))

print("\nNumber of mismatched records:")

print(
    (~verification["both_values_match"]).sum()
)

print("\nOutput files:")
print(DETAIL_FILE)
print(SUMMARY_FILE)

print("\nVerification completed.")

**[orig #244]** Full event inventory vs rebuilt grid extraction audit  
<sub>• output not saved in the source notebook</sub>

In [ ]:
# ============================================================
# FULL EVENT INVENTORY VS REBUILT GRID EXTRACTION AUDIT
# 3 MODELS | ALL NON-SENTINEL REVERSAL EVENTS
# ============================================================

import os
import pandas as pd
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

EVENT_FILE = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

GRID_DIR = os.path.join(
    BASE_DIR,
    "Alwar_daily_extracted_REBUILT"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Cross_Model_Synchronisation_QC"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# LOAD EVENTS
# ------------------------------------------------------------

events = pd.read_csv(EVENT_FILE)

events["Date"] = pd.to_datetime(
    events["Date"]
).dt.normalize()

models = [
    "CanESM5",
    "INM-CM4-8",
    "INM-CM5-0"
]

events = events[
    events["model"].isin(models)
].copy()

print("Total events to verify:", len(events))

# ------------------------------------------------------------
# VERIFY EACH MODEL
# ------------------------------------------------------------

all_results = []

for model in models:

    print("\nProcessing:", model)

    model_events = events[
        events["model"] == model
    ].copy()

    tmax_file = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMaxData_Alwar_REBUILT.csv"
    )

    tmin_file = os.path.join(
        GRID_DIR,
        f"{model}_ssp245_TMinData_Alwar_REBUILT.csv"
    )

    if not os.path.exists(tmax_file):
        raise FileNotFoundError(tmax_file)

    if not os.path.exists(tmin_file):
        raise FileNotFoundError(tmin_file)

    tmax = pd.read_csv(tmax_file)
    tmin = pd.read_csv(tmin_file)

    tmax["Date"] = pd.to_datetime(
        tmax["Date"]
    ).dt.normalize()

    tmin["Date"] = pd.to_datetime(
        tmin["Date"]
    ).dt.normalize()

    tmax = tmax.set_index("Date")
    tmin = tmin.set_index("Date")

    # Validate event grid columns
    model_events["grid_column"] = (
        "grid_col_" +
        model_events["grid_index"].astype(int).astype(str)
    )

    missing_columns = (
        set(model_events["grid_column"])
        - set(tmax.columns)
    )

    missing_tmin_columns = (
        set(model_events["grid_column"])
        - set(tmin.columns)
    )

    if missing_columns or missing_tmin_columns:
        raise ValueError(
            f"Missing columns for {model}: "
            f"Tmax={missing_columns}, Tmin={missing_tmin_columns}"
        )

    # Use a date/grid-cell lookup for each event
    for _, event in model_events.iterrows():

        date = event["Date"]
        column = event["grid_column"]

        if date not in tmax.index or date not in tmin.index:
            raise ValueError(
                f"Missing date: {model}, {date}"
            )

        tx = float(tmax.loc[date, column])
        tn = float(tmin.loc[date, column])

        event_tx = float(event["Tmax_C"])
        event_tn = float(event["Tmin_C"])

        all_results.append({

            "model": model,

            "Date": date,

            "grid_index": int(event["grid_index"]),

            "grid_column": column,

            "event_Tmax_C": event_tx,

            "extracted_Tmax_C": tx,

            "event_Tmin_C": event_tn,

            "extracted_Tmin_C": tn,

            "Tmax_difference_C": tx - event_tx,

            "Tmin_difference_C": tn - event_tn,

            "Tmax_match": abs(tx - event_tx) <= 1e-6,

            "Tmin_match": abs(tn - event_tn) <= 1e-6,

            "event_reversal": event_tn > event_tx,

            "extracted_reversal": tn > tx

        })

# ------------------------------------------------------------
# FULL VERIFICATION RESULTS
# ------------------------------------------------------------

verification = pd.DataFrame(all_results)

verification["both_values_match"] = (
    verification["Tmax_match"]
    & verification["Tmin_match"]
)

verification["reversal_condition_matches"] = (
    verification["event_reversal"]
    == verification["extracted_reversal"]
)

DETAIL_FILE = os.path.join(
    OUTPUT_DIR,
    "All_Reversal_Events_Extraction_Verification.csv"
)

verification.to_csv(
    DETAIL_FILE,
    index=False
)

# ------------------------------------------------------------
# MODEL SUMMARY
# ------------------------------------------------------------

summary = (
    verification.groupby("model")
    .agg(
        events_checked=("grid_index", "count"),
        Tmax_matches=("Tmax_match", "sum"),
        Tmin_matches=("Tmin_match", "sum"),
        complete_value_matches=("both_values_match", "sum"),
        reversal_condition_matches=(
            "reversal_condition_matches", "sum"
        ),
        maximum_Tmax_difference_C=(
            "Tmax_difference_C",
            lambda x: x.abs().max()
        ),
        maximum_Tmin_difference_C=(
            "Tmin_difference_C",
            lambda x: x.abs().max()
        )
    )
    .reset_index()
)

summary["matching_percentage"] = (
    100 * summary["complete_value_matches"]
    / summary["events_checked"]
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "All_Reversal_Events_Extraction_Verification_Summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------
# FINAL AUDIT
# ------------------------------------------------------------

print("\n============================================")
print("FULL EVENT EXTRACTION AUDIT COMPLETED")
print("============================================")

display(summary.round(6))

print("\nTotal verified records:", len(verification))

print(
    "Total mismatches:",
    (~verification["both_values_match"]).sum()
)

print(
    "Reversal-condition mismatches:",
    (~verification["reversal_condition_matches"]).sum()
)

print("\nDetailed report:", DETAIL_FILE)
print("Summary report:", SUMMARY_FILE)

**[orig #247]** Original-source validation of all 2,804 reversal events (corrected file structure)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# ORIGINAL CMIP6 SOURCE VALIDATION - CORRECTED FILE STRUCTURE
# Verify all 2,804 temperature reversal events
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

SOURCE_BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/India"
)

EVENT_FILE = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Original_Source_Validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = ["CanESM5", "INM-CM4-8", "INM-CM5-0"]
TOLERANCE = 1e-6

# ------------------------------------------------------------
# 2. LOAD EVENT INVENTORY
# ------------------------------------------------------------

events = pd.read_csv(EVENT_FILE)

events["Date"] = pd.to_datetime(
    events["Date"], errors="raise"
).dt.normalize()

events["grid_index"] = pd.to_numeric(
    events["grid_index"], errors="raise"
).astype(int)

events["Tmax_C"] = pd.to_numeric(
    events["Tmax_C"], errors="raise"
)

events["Tmin_C"] = pd.to_numeric(
    events["Tmin_C"], errors="raise"
)

events["model"] = events["model"].astype(str)

events = events[
    events["model"].isin(MODELS)
].copy()

print("Total events:", len(events))
print(events.groupby("model").size())

if len(events) != 2804:
    raise ValueError("Expected exactly 2804 events.")

# ------------------------------------------------------------
# 3. CORRECT ORIGINAL FILE LOCATIONS
# ------------------------------------------------------------

def locate_source_file(model, variable):

    file_path = os.path.join(
        SOURCE_BASE,
        model,
        "ssp245",
        variable
    )

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Original source file not found: {file_path}"
        )

    return file_path


# ------------------------------------------------------------
# 4. READ REQUIRED ORIGINAL SOURCE VALUES
# ------------------------------------------------------------

def read_original_event_values(
    file_path,
    required_dates,
    required_grids
):

    results = {}
    duplicate_dates = set()
    malformed_rows = []

    required_dates = set(required_dates)
    required_grids = set(required_grids)

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        # Skip the two metadata rows
        next(f, None)
        next(f, None)

        for line_number, line in enumerate(f, start=3):

            if not line.strip():
                continue

            parts = line.split()

            if len(parts) < 4:
                continue

            try:
                year = int(float(parts[0]))
                month = int(float(parts[1]))
                day = int(float(parts[2]))

                current_date = pd.Timestamp(
                    year=year,
                    month=month,
                    day=day
                ).normalize()

            except Exception:
                continue

            if current_date not in required_dates:
                continue

            if current_date in results:
                duplicate_dates.add(current_date)

            values = {}

            for grid_index in required_grids:

                # First 3 fields are Year, Month, Day
                source_position = 3 + grid_index

                if source_position >= len(parts):
                    malformed_rows.append({
                        "date": current_date,
                        "line_number": line_number,
                        "grid_index": grid_index,
                        "source_position": source_position,
                        "available_fields": len(parts)
                    })
                    continue

                try:
                    values[grid_index] = float(
                        parts[source_position]
                    )

                except Exception:
                    malformed_rows.append({
                        "date": current_date,
                        "line_number": line_number,
                        "grid_index": grid_index,
                        "source_position": source_position,
                        "available_fields": len(parts)
                    })

            results[current_date] = values

    return results, duplicate_dates, malformed_rows


# ------------------------------------------------------------
# 5. VERIFY EACH MODEL
# ------------------------------------------------------------

all_results = []
source_inventory = []
source_issues = []
duplicate_date_report = []

for model in MODELS:

    print("\n" + "=" * 65)
    print("PROCESSING:", model)
    print("=" * 65)

    model_events = events[
        events["model"] == model
    ].copy()

    required_dates = set(model_events["Date"])

    required_grids = set(
        model_events["grid_index"].astype(int)
    )

    tmax_file = locate_source_file(model, "TMaxData")
    tmin_file = locate_source_file(model, "TMinData")

    print("Tmax:", tmax_file)
    print("Tmin:", tmin_file)

    source_inventory.extend([
        {
            "model": model,
            "variable": "Tmax",
            "source_file": tmax_file
        },
        {
            "model": model,
            "variable": "Tmin",
            "source_file": tmin_file
        }
    ])

    tmax_data, tmax_duplicates, tmax_issues = (
        read_original_event_values(
            tmax_file,
            required_dates,
            required_grids
        )
    )

    tmin_data, tmin_duplicates, tmin_issues = (
        read_original_event_values(
            tmin_file,
            required_dates,
            required_grids
        )
    )

    print("Required dates:", len(required_dates))
    print("Tmax dates found:", len(tmax_data))
    print("Tmin dates found:", len(tmin_data))

    for issue in tmax_issues:
        source_issues.append({
            "model": model,
            "variable": "Tmax",
            **issue
        })

    for issue in tmin_issues:
        source_issues.append({
            "model": model,
            "variable": "Tmin",
            **issue
        })

    duplicate_date_report.extend([
        {
            "model": model,
            "variable": "Tmax",
            "duplicate_date_count": len(tmax_duplicates)
        },
        {
            "model": model,
            "variable": "Tmin",
            "duplicate_date_count": len(tmin_duplicates)
        }
    ])

    # Event-by-event comparison
    for _, event in model_events.iterrows():

        date = event["Date"]
        grid = int(event["grid_index"])

        expected_tmax = float(event["Tmax_C"])
        expected_tmin = float(event["Tmin_C"])

        source_tmax = (
            tmax_data.get(date, {}).get(grid, np.nan)
        )

        source_tmin = (
            tmin_data.get(date, {}).get(grid, np.nan)
        )

        tmax_found = np.isfinite(source_tmax)
        tmin_found = np.isfinite(source_tmin)

        tmax_match = (
            tmax_found
            and abs(source_tmax - expected_tmax) <= TOLERANCE
        )

        tmin_match = (
            tmin_found
            and abs(source_tmin - expected_tmin) <= TOLERANCE
        )

        complete_match = tmax_match and tmin_match

        inventory_reversal = expected_tmax < expected_tmin

        source_reversal = (
            source_tmax < source_tmin
            if tmax_found and tmin_found
            else False
        )

        reversal_match = (
            source_reversal == inventory_reversal
            if tmax_found and tmin_found
            else False
        )

        all_results.append({

            "model": model,
            "Date": date,
            "grid_index": grid,

            "inventory_Tmax_C": expected_tmax,
            "inventory_Tmin_C": expected_tmin,

            "source_Tmax_C": source_tmax,
            "source_Tmin_C": source_tmin,

            "Tmax_found": tmax_found,
            "Tmin_found": tmin_found,

            "Tmax_match": tmax_match,
            "Tmin_match": tmin_match,

            "complete_value_match": complete_match,

            "inventory_reversal": inventory_reversal,
            "source_reversal": source_reversal,

            "reversal_condition_match": reversal_match,

            "Tmax_difference_C": (
                source_tmax - expected_tmax
                if tmax_found else np.nan
            ),

            "Tmin_difference_C": (
                source_tmin - expected_tmin
                if tmin_found else np.nan
            )
        })

    print("Events processed:", len(model_events))


# ------------------------------------------------------------
# 6. SAVE DETAILED REPORT
# ------------------------------------------------------------

detail = pd.DataFrame(all_results)

detail_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_All_2804_Event_Verification.csv"
)

detail.to_csv(detail_file, index=False)


# ------------------------------------------------------------
# 7. MODEL SUMMARY
# ------------------------------------------------------------

summary_rows = []

for model in MODELS:

    subset = detail[
        detail["model"] == model
    ]

    summary_rows.append({

        "model": model,
        "events_checked": len(subset),

        "Tmax_found": int(subset["Tmax_found"].sum()),
        "Tmin_found": int(subset["Tmin_found"].sum()),

        "Tmax_matches": int(subset["Tmax_match"].sum()),
        "Tmin_matches": int(subset["Tmin_match"].sum()),

        "complete_value_matches": int(
            subset["complete_value_match"].sum()
        ),

        "reversal_condition_matches": int(
            subset["reversal_condition_match"].sum()
        ),

        "complete_mismatches": int(
            (~subset["complete_value_match"]).sum()
        ),

        "maximum_Tmax_difference_C": (
            subset["Tmax_difference_C"].abs().max()
        ),

        "maximum_Tmin_difference_C": (
            subset["Tmin_difference_C"].abs().max()
        ),

        "matching_percentage": (
            100 * subset["complete_value_match"].mean()
        )
    })

summary = pd.DataFrame(summary_rows)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_All_2804_Event_Summary.csv"
)

summary.to_csv(summary_file, index=False)


# ------------------------------------------------------------
# 8. SAVE SOURCE INVENTORY AND ISSUES
# ------------------------------------------------------------

pd.DataFrame(source_inventory).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_File_Inventory.csv"
    ),
    index=False
)

pd.DataFrame(source_issues).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Row_Parsing_Issues.csv"
    ),
    index=False
)

pd.DataFrame(duplicate_date_report).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Duplicate_Date_Report.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 9. FINAL AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ORIGINAL SOURCE VALIDATION SUMMARY")
print("=" * 75)

display(summary)

print("\nTotal events:", len(detail))

print(
    "Complete value matches:",
    int(detail["complete_value_match"].sum())
)

print(
    "Complete value mismatches:",
    int((~detail["complete_value_match"]).sum())
)

print(
    "Reversal condition matches:",
    int(detail["reversal_condition_match"].sum())
)

print(
    "Reversal condition mismatches:",
    int((~detail["reversal_condition_match"]).sum())
)

print(
    "Missing Tmax:",
    int((~detail["Tmax_found"]).sum())
)

print(
    "Missing Tmin:",
    int((~detail["Tmin_found"]).sum())
)

print("Source parsing issues:", len(source_issues))

audit_pass = (
    len(detail) == 2804
    and detail["complete_value_match"].all()
    and detail["reversal_condition_match"].all()
    and detail["Tmax_found"].all()
    and detail["Tmin_found"].all()
    and len(source_issues) == 0
)

print("\nFINAL AUDIT STATUS:")

if audit_pass:
    print("PASS: All 2,804 events match the original source files.")
else:
    print("REVIEW REQUIRED: Inspect detailed reports.")

print("\nReports saved in:")
print(OUTPUT_DIR)


**[orig #248]** Sentinel value audit: −273.15 °C in the original source, all 13 models  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ==============================================================
# ORIGINAL-SOURCE SENTINEL VALUE AUDIT
# All 13 CMIP6 models | Alwar's 22 selected grid cells
# Candidate value: -273.15 C
# ==============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------
# 1. PATHS AND SETTINGS
# --------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

SOURCE_BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/India"
)

WEIGHT_FILE = os.path.join(
    BASE_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Original_Source_Validation",
    "Sentinel_Audit"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

VARIABLES = {
    "Tmax": "TMaxData",
    "Tmin": "TMinData"
}

SENTINEL = -273.15
TOLERANCE = 1e-6

# --------------------------------------------------------------
# 2. LOAD ALWAR GRID INDICES
# --------------------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

grid_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

print("Unique Alwar grid cells:", len(grid_indices))
print("Grid indices:", grid_indices)

if len(grid_indices) != 22:
    raise ValueError(
        f"Expected 22 Alwar grid cells, found {len(grid_indices)}"
    )

# --------------------------------------------------------------
# 3. INSPECT AVAILABLE ARCHIVE DOCUMENTATION
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("ARCHIVE DOCUMENTATION / FILE INVENTORY")
print("=" * 70)

DOC_FILES = [
    os.path.join(
        os.path.dirname(SOURCE_BASE),
        "India_archive_file_listing.txt"
    ),
    os.path.join(
        os.path.dirname(SOURCE_BASE),
        "India_climate_filename_matches.csv"
    )
]

for doc_file in DOC_FILES:

    print("\nFILE:", doc_file)

    if not os.path.exists(doc_file):
        print("Not found")
        continue

    if doc_file.endswith(".csv"):
        doc = pd.read_csv(doc_file)
        print(doc.head(15).to_string(index=False))
        print("Columns:", doc.columns.tolist())

    else:
        with open(
            doc_file,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:
            for i, line in enumerate(f):
                if i >= 25:
                    break
                print(line.rstrip())

# --------------------------------------------------------------
# 4. SOURCE FILE READER
# --------------------------------------------------------------

def source_file_path(model, variable):

    return os.path.join(
        SOURCE_BASE,
        model,
        "ssp245",
        VARIABLES[variable]
    )


def audit_source_file(file_path, model, variable):

    if not os.path.isfile(file_path):
        raise FileNotFoundError(file_path)

    records = []
    daily_rows = 0
    malformed_dates = 0
    short_rows = 0

    # Inspect metadata rows separately
    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        metadata_1 = next(f, "").strip()
        metadata_2 = next(f, "").strip()

        print(
            f"\n{model} | {variable} | "
            f"Metadata row lengths: "
            f"{len(metadata_1.split())}, "
            f"{len(metadata_2.split())}"
        )

        for line_number, line in enumerate(f, start=3):

            if not line.strip():
                continue

            parts = line.split()

            if len(parts) < 4:
                short_rows += 1
                continue

            try:
                year = int(float(parts[0]))
                month = int(float(parts[1]))
                day = int(float(parts[2]))

                date = pd.Timestamp(
                    year=year,
                    month=month,
                    day=day
                )

            except Exception:
                malformed_dates += 1
                continue

            daily_rows += 1

            # Read only Alwar's 22 selected grid cells
            for grid_index in grid_indices:

                source_position = 3 + grid_index

                if source_position >= len(parts):
                    short_rows += 1
                    continue

                try:
                    value = float(parts[source_position])
                except Exception:
                    continue

                if (
                    np.isfinite(value)
                    and abs(value - SENTINEL) <= TOLERANCE
                ):

                    records.append({
                        "model": model,
                        "variable": variable,
                        "date": date,
                        "year": date.year,
                        "month": date.month,
                        "day": date.day,
                        "grid_index": grid_index,
                        "value_C": value,
                        "source_line_number": line_number
                    })

    diagnostics = {
        "model": model,
        "variable": variable,
        "daily_rows": daily_rows,
        "sentinel_occurrences": len(records),
        "malformed_dates": malformed_dates,
        "short_rows": short_rows,
        "source_file": file_path
    }

    return records, diagnostics

# --------------------------------------------------------------
# 5. AUDIT ALL 13 MODELS AND BOTH VARIABLES
# --------------------------------------------------------------

all_sentinel_records = []
all_diagnostics = []

for model in MODELS:

    print("\n" + "#" * 70)
    print("MODEL:", model)
    print("#" * 70)

    for variable in ["Tmax", "Tmin"]:

        path = source_file_path(model, variable)

        print("\nAuditing:", variable)

        records, diagnostics = audit_source_file(
            path,
            model,
            variable
        )

        all_sentinel_records.extend(records)
        all_diagnostics.append(diagnostics)

        print(
            "Daily rows:",
            diagnostics["daily_rows"]
        )

        print(
            "Sentinel occurrences:",
            diagnostics["sentinel_occurrences"]
        )

# --------------------------------------------------------------
# 6. CREATE DETAILED SENTINEL INVENTORY
# --------------------------------------------------------------

sentinel_df = pd.DataFrame(
    all_sentinel_records
)

if sentinel_df.empty:
    sentinel_df = pd.DataFrame(columns=[
        "model", "variable", "date", "year",
        "month", "day", "grid_index",
        "value_C", "source_line_number"
    ])

detail_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_Sentinel_Event_Inventory.csv"
)

sentinel_df.to_csv(detail_file, index=False)

# --------------------------------------------------------------
# 7. MODEL-WISE SUMMARY
# --------------------------------------------------------------

diagnostics_df = pd.DataFrame(all_diagnostics)

summary = (
    diagnostics_df.pivot(
        index="model",
        columns="variable",
        values="sentinel_occurrences"
    )
    .reset_index()
)

summary.columns.name = None

for variable in ["Tmax", "Tmin"]:
    if variable not in summary.columns:
        summary[variable] = 0

summary["Tmax"] = summary["Tmax"].fillna(0).astype(int)
summary["Tmin"] = summary["Tmin"].fillna(0).astype(int)

summary["Total"] = summary["Tmax"] + summary["Tmin"]

summary_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_Sentinel_Model_Summary.csv"
)

summary.to_csv(summary_file, index=False)

# --------------------------------------------------------------
# 8. GRID-CELL SUMMARY
# --------------------------------------------------------------

if len(sentinel_df):

    grid_summary = (
        sentinel_df.groupby(
            ["model", "variable", "grid_index"]
        )
        .agg(
            occurrences=("value_C", "size"),
            unique_dates=("date", "nunique")
        )
        .reset_index()
    )

else:

    grid_summary = pd.DataFrame(
        columns=[
            "model", "variable", "grid_index",
            "occurrences", "unique_dates"
        ]
    )

grid_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_Sentinel_Grid_Summary.csv"
)

grid_summary.to_csv(grid_file, index=False)

# --------------------------------------------------------------
# 9. MONTHLY AND ANNUAL DISTRIBUTION
# --------------------------------------------------------------

if len(sentinel_df):

    monthly_summary = (
        sentinel_df.groupby(
            ["model", "variable", "month"]
        )
        .size()
        .reset_index(name="occurrences")
    )

    annual_summary = (
        sentinel_df.groupby(
            ["model", "variable", "year"]
        )
        .size()
        .reset_index(name="occurrences")
    )

else:

    monthly_summary = pd.DataFrame()
    annual_summary = pd.DataFrame()

monthly_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Sentinel_Monthly_Summary.csv"
    ),
    index=False
)

annual_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Sentinel_Annual_Summary.csv"
    ),
    index=False
)

# --------------------------------------------------------------
# 10. DATA-READING DIAGNOSTICS
# --------------------------------------------------------------

diagnostics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Sentinel_Read_Diagnostics.csv"
    ),
    index=False
)

# --------------------------------------------------------------
# 11. COMPARE WITH PREVIOUS EXTRACTED-DATA COUNTS
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL SOURCE SENTINEL AUDIT RESULTS")
print("=" * 70)

display(summary)

total_tmax = int(summary["Tmax"].sum())
total_tmin = int(summary["Tmin"].sum())

print("\nOriginal-source Tmax sentinel occurrences:", total_tmax)
print("Original-source Tmin sentinel occurrences:", total_tmin)
print("Original-source combined occurrences:", total_tmax + total_tmin)

print("\nPreviously reported extracted Alwar counts:")
print("Tmax: 13")
print("Tmin: 171")

print("\nCount comparison:")
print("Tmax difference:", total_tmax - 13)
print("Tmin difference:", total_tmin - 171)

print("\nTotal daily records read:")
print(int(diagnostics_df["daily_rows"].sum()))

print(
    "Malformed dates:",
    int(diagnostics_df["malformed_dates"].sum())
)

print(
    "Short source rows:",
    int(diagnostics_df["short_rows"].sum())
)

print("\nDetailed inventory:")
print(detail_file)

print("\nSummary:")
print(summary_file)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nAudit completed.")
